# Bearing-UAV · iMatch · MEAN · Sample4Geo — 새 Kaggle 실행
이전 노트북, RUN, manifest, 결과 CSV 없이 시작합니다. 학습 없이 공개 pretrained 가중치를 사용합니다.

1. Kaggle에서 이 ipynb를 Import하고 GPU와 Internet을 켭니다.
2. `visloc-single-eval` 이미지/CSV 데이터셋을 Add Input으로 연결합니다.
3. 공통 셀 1~4를 실행합니다.
4. 원하는 모델의 **pretrained 준비·3장 확인 셀 → 전체 평가 셀**을 실행합니다.
   네 모델은 서로 실행 결과에 의존하지 않습니다. 전체 실행도 가능합니다.
5. 마지막 결과 저장 셀에서 비교표와 ZIP을 받습니다. 세션 종료 전 Save Version으로 Output을 저장하세요.

같은 세션에서 설정 셀을 다시 실행해도 RUN을 유지합니다. 커널 재시작 시에는
`RESUME_RUN`에 남아 있는 결과 폴더를 지정하고 공통 셀부터 실행하세요.
이전 결과가 없어도 기본값으로 새 실행을 시작합니다.

Bearing-UAV/iMatch는 UAV–지도 블록 내 위치 추정, MEAN/Sample4Geo는 전체 갤러리 검색입니다.
현재 GPS 중심 크롭은 위치 순위 평가용이 아닙니다. 검색 모델의 공통 좌표가 없으면
미터 정확도·Recall@1은 N/A이며, 검색 ID/유사도/시간은 저장합니다.
검색 모델의 갤러리 특징 사전 계산 시간은 별도 기록하므로 네 모델의 지연 시간 범위를 확인하세요.

## 1. 새 실행 설정

In [3]:
from pathlib import Path
import sys, os, json, subprocess, datetime, hashlib, shutil, traceback

INPUT_ROOT = Path('/kaggle/input')
SAMPLE_CSV = None  # 여러 호환 CSV가 있으면 정확한 CSV 경로 지정
LIMIT = None      # 전체 데이터. 빠른 확인만 하려면 5 (새 RUN 사용)
WARMUP, REPEATS = 5, 3
METERS_PER_UNIT = 32.0
ANGLES, STOP_INLIERS = 8, 15
DIAGNOSTIC_ONLY = True
RESUME_RUN = ''  # 커널 재시작 후 남아 있는 run 폴더를 지정할 때만 사용
WORK = Path('/kaggle/working/four_model_benchmark')
WORK.mkdir(parents=True, exist_ok=True)
if RESUME_RUN:
    RUN = Path(RESUME_RUN)
    if not RUN.is_dir() or not RUN.resolve().is_relative_to(Path('/kaggle/working').resolve()):
        raise ValueError('RESUME_RUN must be an existing writable run under /kaggle/working')
elif 'RUN' not in globals() or not Path(RUN).is_relative_to(WORK):
    RUN = WORK / ('run_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    RUN.mkdir()
SOURCE = RUN/'source'
SOURCE.mkdir(exist_ok=True)
CACHE = WORK/'pretrained'
CACHE.mkdir(exist_ok=True)
print('결과 폴더:', RUN)
print('데이터 입력:', [str(p) for p in INPUT_ROOT.iterdir()])

결과 폴더: /kaggle/working/four_model_benchmark/run_20261114T024104439388Z
데이터 입력: ['/kaggle/input/datasets']


## 2. 공통 환경 및 모델별 격리 실행 도구

In [4]:
import pandas as pd
import numpy as np
import torch
import cv2
from IPython.display import display, FileLink
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle Settings에서 GPU accelerator를 활성화하세요.')
DEVICE = 'cuda:0'
MODELS = ['Bearing-UAV', 'iMatch', 'MEAN', 'Sample4Geo']

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024**2), b''): h.update(chunk)
    return h.hexdigest()

def make_env(name, packages):
    import sys
    import importlib.metadata as metadata
    env=WORK/'envs'/name
    py=env/'bin/python'
    env.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/(name+'_environment.log')

    def command(args, required=True):
        result=subprocess.run([str(x) for x in args],stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True)
        output=result.stdout or ''
        with log.open('a',encoding='utf-8') as f:
            f.write(repr([str(x) for x in args])+chr(10)+output+chr(10))
        if output: print(output,flush=True)
        if required and result.returncode:
            raise RuntimeError(f'Environment command failed ({result.returncode}). See {log}: '+output[-4000:])
        return result

    def valid_python():
        if not py.exists(): return False
        try:
            probe=command([py,'-c',
                'import sys; from pathlib import Path; '
                'assert Path(sys.prefix).resolve()==Path(sys.argv[1]).resolve(); '
                'assert sys.prefix != sys.base_prefix',env],required=False)
            return probe.returncode==0
        except OSError:
            return False

    if not valid_python():
        # Kaggle may omit ensurepip. Create the interpreter without invoking it.
        created=command([sys.executable,'-m','venv','--without-pip',
                         '--system-site-packages',env],required=False)
        if created.returncode or not valid_python():
            # Fallback also handles installations without the stdlib venv module.
            bootstrap=WORK/'virtualenv_bootstrap'
            command([sys.executable,'-m','pip','install','--target',bootstrap,'virtualenv'])
            launcher='import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module("virtualenv",run_name="__main__")'
            command([sys.executable,'-c',launcher,bootstrap,'--system-site-packages',env])
        if not valid_python():
            raise RuntimeError(f'Invalid environment interpreter: {py}. See {log}')
    # Parent pip can install into a pip-less environment; no ensurepip needed.
    pip_check=command([py,'-m','pip','--version'],required=False)
    if pip_check.returncode:
        command([sys.executable,'-m','pip','--python',py,'install','pip'])
    command([py,'-m','pip','--version'])
    pins=[package+'=='+metadata.version(package) for package in ('torch','torchvision','numpy')]
    constraint=env/'constraints.txt'
    constraint.write_text(chr(10).join(pins),encoding='utf-8')
    command([py,'-m','pip','install','-c',constraint,*packages])
    frozen=command([py,'-m','pip','freeze'])
    (RUN/(name+'_pip-freeze.txt')).write_text(frozen.stdout,encoding='utf-8')
    return py

def run_worker(name, py, worker, config):
    tag='smoke' if config['smoke'] else 'full'
    config_path=RUN/(name+'_'+tag+'_config.json')
    save_json(config_path,config)
    log=RUN/(name+'_'+tag+'.log')
    with log.open('w',encoding='utf-8') as f:
        proc=subprocess.Popen([str(py),'-u',str(worker),str(config_path)],
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                print(line,end='',flush=True); f.write(line); f.flush()
            rc=proc.wait()
        finally:
            if proc.poll() is None:
                proc.terminate(); proc.wait()
            proc.stdout.close()
    if rc:
        raise RuntimeError(f'{name} {tag} failed. Log: {log}')
    return json.loads((Path(config['output'])/'summary.json').read_text(encoding='utf-8'))

print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__)
save_json(RUN/'environment.json',dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),warmup=WARMUP,repeats=REPEATS,meters_per_unit=METERS_PER_UNIT))

GPU: Tesla T4 torch: 2.11.0+cu130


## 3. 내장 평가 코드 설치 및 원본 데이터 로딩

In [7]:
import benchmark_core as bc

if not hasattr(bc, "_original_sha256"):
    bc._original_sha256 = bc.sha256

_hash_progress = {"count": 0}

def sha256_with_progress(path):
    n = _hash_progress["count"] + 1
    if n == 1 or n % 25 == 0:
        print(f"[이미지 검사] {n}번째 파일 읽는 중: {path}", flush=True)
    result = bc._original_sha256(path)
    _hash_progress["count"] = n
    return result

bc.sha256 = sha256_with_progress
print("진행 로그 적용 완료. 셀 3을 다시 실행하세요.")

진행 로그 적용 완료. 셀 3을 다시 실행하세요.


In [8]:
SOURCES = {'benchmark_core.py': '"""Common single-image localization protocol. No ground truth reaches adapters."""\nfrom pathlib import Path\nimport gc\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport random\nimport time\nimport csv\nimport io\nimport zipfile\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for chunk in iter(lambda: f.read(2**20), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef save_json(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef discover_sample_csv(root, extract_dir=None, fallback_root=None):\n    """Find by schema, not Kaggle slug layout. Refuse ambiguous datasets."""\n    required = {\'set\', \'k\', \'x_norm\', \'y_norm\'}\n    root = Path(root)\n    def compatible(header):\n        return required.issubset(next(csv.reader([header.lstrip(\'\\ufeff\')]), []))\n    def scan(folder):\n        candidates = []\n        for p in sorted(folder.rglob(\'*.csv\')):\n            try:\n                with p.open(encoding=\'utf-8-sig\') as f:\n                    if compatible(f.readline()):\n                        candidates.append(p)\n            except (UnicodeError, OSError, csv.Error):\n                continue\n        return candidates\n    paths = scan(root) if root.exists() else []\n    if not paths and fallback_root and Path(fallback_root).exists() and root != Path(fallback_root):\n        print(f\'No compatible CSV under {root}; searching {fallback_root}\')\n        root = Path(fallback_root)\n        paths = scan(root)\n    if len(paths) > 1:\n        raise ValueError(f\'Multiple compatible CSVs. Set SAMPLE_CSV explicitly: {[str(p) for p in paths]}\')\n    if paths:\n        return paths[0]\n    archives = []\n    for archive in sorted(root.rglob(\'*.zip\')):\n        with zipfile.ZipFile(archive) as z:\n            for member in z.infolist():\n                if member.filename.lower().endswith(\'.csv\'):\n                    try:\n                        with z.open(member) as f:\n                            header = f.readline(65536).decode(\'utf-8-sig\')\n                        if compatible(header):\n                            archives.append((archive, member.filename))\n                    except (UnicodeError, csv.Error):\n                        continue\n    if len(archives) > 1:\n        raise ValueError(f\'Multiple datasets inside ZIPs; select/extract one explicitly: {archives}\')\n    if archives:\n        if extract_dir is None:\n            raise ValueError(f\'Dataset is zipped: {archives[0]}. Set extract_dir to a writable folder.\')\n        archive, member_name = archives[0]\n        destination = Path(extract_dir) / sha256(archive)[:16]\n        destination.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(archive) as z:\n            for member in z.infolist():\n                if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):\n                    raise ValueError(f\'Unsafe ZIP member: {member.filename}\')\n            z.extractall(destination)\n        return destination / member_name\n    inventory = [str(p.relative_to(root)) for p in root.rglob(\'*\') if p.is_file()][:60]\n    raise FileNotFoundError(f\'No CSV with columns {sorted(required)} under {root}. \'\n                            f\'Input files (first 60): {inventory}. Raw VisLoc data needs conversion; labels are not inferred.\')\n\n\ndef read_dataset(root, sample_csv=None, limit=None, extract_dir=None, fallback_root=None):\n    root = Path(root)\n    paths = [Path(sample_csv)] if sample_csv else [discover_sample_csv(root, extract_dir, fallback_root)]\n    table = pd.read_csv(paths[0])\n    required = {\'set\', \'k\', \'x_norm\', \'y_norm\'}\n    if not required.issubset(table.columns):\n        raise ValueError(f\'Missing columns: {required - set(table.columns)}\')\n    if table.empty or table.duplicated([\'set\', \'k\']).any():\n        raise ValueError(\'Empty dataset or duplicate (set,k) identifiers\')\n    table = table.sort_values([\'set\', \'k\']).reset_index(drop=True)\n    if limit is not None:\n        if limit < 1:\n            raise ValueError(\'limit must be positive\')\n        table = table.iloc[:limit].copy()\n    if not np.isfinite(table[[\'x_norm\', \'y_norm\']].to_numpy(float)).all():\n        raise ValueError(\'Nonfinite ground truth\')\n    base = paths[0].parent\n    records = []\n    for r in table.to_dict(\'records\'):\n        item = {\'sample_id\': f"{r[\'set\']}:{int(r[\'k\'])}", \'set\': r[\'set\'], \'k\': int(r[\'k\']),\n                \'gt_x\': float(r[\'x_norm\']), \'gt_y\': float(r[\'y_norm\'])}\n        for kind in (\'uav\', \'block\'):\n            p = base / str(r.get(f\'{kind}_path\', f"{r[\'set\']}/k{int(r[\'k\']):03d}_{kind}.jpg"))\n            if not p.is_file():\n                raise FileNotFoundError(p)\n            item[f\'{kind}_path\'] = str(p)\n            item[f\'{kind}_sha256\'] = sha256(p)\n        records.append(item)\n    return pd.DataFrame(records), paths[0]\n\n\ndef dataset_audit(manifest, meters_per_unit):\n    e = np.hypot(manifest.gt_x, manifest.gt_y) * meters_per_unit\n    centered = bool(np.allclose(manifest[[\'gt_x\', \'gt_y\']], 0, atol=1e-6))\n    return {\'n\': len(manifest), \'all_labels_at_center\': centered,\n            \'center_baseline_accuracy_10m_pct\': float((e < 10).mean() * 100),\n            \'position_ranking_valid\': not centered,\n            \'note\': \'GPS-centered crops reveal the target location; position ranking is diagnostic only.\' if centered\n                    else \'Nonconstant labels do not prove absence of leakage; audit crop generation and train/test overlap.\'}\n\n\ndef load_pair(row):\n    images = []\n    for key, shape in [(\'uav_path\', (256, 256)), (\'block_path\', (512, 512))]:\n        bgr = cv2.imread(row[key])\n        if bgr is None or bgr.shape[:2] != shape:\n            raise ValueError(f\'{row[key]}: expected {shape}, got {None if bgr is None else bgr.shape}\')\n        images.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))\n    return images\n\n\ndef position_error(prediction, gx, gy, meters_per_unit):\n    x, y = float(prediction.get(\'x\', float(\'nan\'))), float(prediction.get(\'y\', float(\'nan\')))\n    valid = bool(prediction.get(\'valid\', True)) and math.isfinite(x) and math.isfinite(y)\n    # Missing/rejected predictions are failures, including when the GT is exactly (0,0).\n    # No clipping: clipping toward GT can inflate accuracy on wrong map blocks.\n    error = math.hypot(x - gx, y - gy) * meters_per_unit if valid else float(\'inf\')\n    return x, y, valid, error\n\n\ndef summarize(predictions, params, audit, metadata):\n    e = predictions.error_m.to_numpy(float)\n    times = predictions.seconds.to_numpy(float)\n    return {**metadata, \'status\': \'measured\', \'n\': len(e),\n            \'accuracy_10m_pct\': float((e < 10).mean() * 100),\n            \'accuracy_20m_pct\': float((e < 20).mean() * 100),\n            \'accuracy_32m_pct\': float((e < 32).mean() * 100),\n            \'valid_pct\': float(predictions.valid.mean() * 100),\n            \'mean_error_valid_m\': float(e[np.isfinite(e)].mean()) if np.isfinite(e).any() else np.nan,\n            \'mean_ms\': float(times.mean() * 1000), \'p50_ms\': float(np.median(times) * 1000),\n            \'p95_ms\': float(np.percentile(times, 95) * 1000), \'fps\': float(1 / times.mean()),\n            \'total_parameters\': params, \'parameters_M\': params / 1e6,\n            \'position_ranking_valid\': audit[\'position_ranking_valid\']}\n\n\ndef all_modules(roots):\n    """Find nn.Modules in wrappers/plain containers too; shared objects counted once."""\n    seen, modules = set(), []\n    def visit(obj):\n        if id(obj) in seen:\n            return\n        seen.add(id(obj))\n        if isinstance(obj, torch.nn.Module):\n            modules.append(obj)\n            for value in vars(obj).values():\n                visit(value)\n        elif isinstance(obj, dict):\n            for value in obj.values():\n                visit(value)\n        elif isinstance(obj, (tuple, list)):\n            for value in obj:\n                visit(value)\n    visit(roots)\n    return modules\n\n\ndef parameter_count(roots):\n    unique = {}\n    for module in all_modules(roots):\n        for p in module.parameters(recurse=False):\n            unique[id(p)] = p\n    if not unique:\n        raise ValueError(\'No parameters found; provide every learned component in adapter.modules\')\n    return sum(p.numel() for p in unique.values())\n\n\ndef fp32_audit(roots):\n    """Observe Conv/Linear computation during warmup, including hidden modules."""\n    observed, handles = set(), []\n    def capture(module, inputs, output):\n        if isinstance(output, torch.Tensor) and output.is_floating_point():\n            observed.add(str(output.dtype))\n    for module in all_modules(roots):\n        if isinstance(module, (torch.nn.Linear, torch.nn.Conv2d)):\n            handles.append(module.register_forward_hook(capture))\n    return observed, handles\n\n\ndef enforce_fp32(roots):\n    """Disable model-internal AMP as well as converting parameter storage."""\n    from dataclasses import fields, is_dataclass, replace\n    for module in all_modules(roots):\n        module.float()\n        for name in (\'amp\', \'use_amp\'):\n            if hasattr(module, name) and isinstance(getattr(module, name), bool):\n                setattr(module, name, False)\n        if hasattr(module, \'amp_dtype\'):\n            module.amp_dtype = torch.float32\n        cfg = getattr(module, \'cfg\', None)\n        if is_dataclass(cfg):\n            changes = {f.name: False for f in fields(cfg) if f.name in (\'mp\', \'amp\', \'use_amp\')}\n            if changes:\n                module.cfg = replace(cfg, **changes)\n\n\nclass IMatchAdapter:\n    def __init__(self, device, angles=8, stop_inl=15, checkpoint=\'\'):\n        import os\n        os.environ[\'LOMA_FP32\'] = \'1\'\n        from imatch.matchers import get_matcher\n        from imatch.pipeline import localize\n        self.match = get_matcher(\'loma\', device)\n        self.localize = localize\n        self.modules = [self.match.m.matcher]\n        # Full LoMa model, including lazy detector/descriptor, is initialized by warmup.\n        from vismatch.utils import set_device_globals\n        set_device_globals(\'loma\', device, amp_dtype=torch.float32)\n        enforce_fp32(self.modules)\n        if checkpoint:\n            state = torch.load(checkpoint, map_location=\'cpu\', weights_only=True)\n            state = state.get(\'model_state_dict\', state)\n            self.modules[0].load_state_dict(state, strict=True)\n        self.angles = list(np.arange(0, 360, 360 / angles))\n        self.stop_inl = stop_inl\n        self.metadata = {\'model\': \'iMatch\', \'variant\': f\'LoMa-B/{angles}dir/stop={stop_inl}\',\n                         \'weights\': str(checkpoint) or \'official LoMa pretrained (no task-specific training)\',\n                         \'checkpoint_sha256\': sha256(checkpoint) if checkpoint else \'\',\n                         \'training_regime\': \'public-pretrained\' if not checkpoint else \'user-checkpoint: audit training split\'}\n\n    def predict(self, uav, block):\n        # GT argument is deliberately constant and is used only by legacy match diagnostics.\n        records, best = self.localize(self.match, uav, block, (0., 0., 0.),\n                                      angles=self.angles, stop_inl=self.stop_inl)\n        if best is None:\n            return {\'valid\': False}\n        # Single-image benchmark: no R1 dead reckoning; low inlier count is recorded, not silently imputed.\n        return {\'x\': best[\'px\'], \'y\': best[\'py\'], \'valid\': True,\n                \'inliers\': best[\'n_inl\'], \'n_angles\': len(records)}\n\n\nclass BearingAdapter:\n    def __init__(self, repo, weights_dir, device):\n        import sys\n        sys.path.insert(0, str(repo))\n        # Official utils_transform accesses np.complex_ during import on NumPy 2.\n        for name, value in {\'complex_\': np.complex128, \'float_\': np.float64}.items():\n            if name not in np.__dict__:\n                setattr(np, name, value)\n        # imgaug 0.4 imports np.sctypes, removed in NumPy 2. This compatibility alias\n        # affects import-time augmentation setup; evaluation uses only ToTensor/Normalize.\n        if not hasattr(np, \'sctypes\'):\n            np.sctypes = {\'int\': [np.int8, np.int16, np.int32, np.int64],\n                          \'uint\': [np.uint8, np.uint16, np.uint32, np.uint64],\n                          \'float\': [np.float16, np.float32, np.float64],\n                          \'complex\': [np.complex64, np.complex128], \'others\': [bool, object, str, bytes]}\n        from cvphr.models.posaglreg.models import load_config_and_model\n        cls, kw = load_config_and_model(str(weights_dir))\n        self.model = cls(**kw)\n        path = Path(weights_dir) / \'best_model.pth\'\n        state = torch.load(path, map_location=\'cpu\', weights_only=True)\n        self.model.load_state_dict(state[\'model_state_dict\'], strict=True)\n        self.model = self.model.float().to(device).eval()\n        self.device, self.modules = device, [self.model]\n        self.metadata = {\'model\': \'Bearing-UAV\', \'variant\': \'official cross_view\',\n                         \'weights\': str(path), \'checkpoint_sha256\': sha256(path),\n                         \'training_regime\': \'public-pretrained: training-city overlap must be audited\'}\n\n    def predict(self, uav, block):\n        # Official p1,p2,p3,p4 order = TL,TR,BL,BR; fifth image = UAV.\n        patches = [block[y:y+256, x:x+256] for x, y in [(0, 0), (256, 0), (0, 256), (256, 256)]] + [uav]\n        x = np.stack(patches).astype(np.float32) / 255\n        x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n        x = torch.from_numpy(x.transpose(0, 3, 1, 2).copy())[None].to(self.device)\n        pos, _direction = self.model(x)\n        xy = pos[0].float().cpu().numpy()\n        return {\'x\': float(xy[0]), \'y\': float(xy[1]), \'valid\': True}\n\n\ndef load_external_adapter(path, config, device):\n    spec = importlib.util.spec_from_file_location(\'external_\' + Path(path).stem, path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    adapter = module.build(config, device)\n    if not callable(getattr(adapter, \'predict\', None)) or not getattr(adapter, \'modules\', None):\n        raise TypeError(\'Adapter must implement predict(uav_rgb, block_rgb) and expose modules\')\n    for field in (\'model\', \'variant\', \'weights\', \'training_regime\'):\n        if not adapter.metadata.get(field):\n            raise ValueError(f\'Missing provenance: {field}\')\n    return adapter\n\n\ndef benchmark(adapter, manifest, out, audit, device=\'cuda\', meters_per_unit=32., warmup=5, repeats=3):\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    if warmup < 1 or repeats < 1:\n        raise ValueError(\'warmup/repeats must be positive\')\n    def sync():\n        if str(device).startswith(\'cuda\'):\n            torch.cuda.synchronize(device)\n    first = load_pair(manifest.iloc[0])\n    with torch.inference_mode():\n        adapter.predict(*first)  # lazy initialization excluded from timing and parameter count\n        enforce_fp32(adapter.modules)\n        for module in all_modules(adapter.modules):\n            module.eval()\n        for _ in range(warmup):\n            adapter.predict(*first)\n        observed, handles = fp32_audit(adapter.modules)\n        try:\n            adapter.predict(*first)\n        finally:\n            for handle in handles:\n                handle.remove()\n        if observed != {\'torch.float32\'}:\n            raise RuntimeError(f\'FP32 compute audit failed: {observed}\')\n        params = parameter_count(adapter.modules)\n        save_json(out / \'model.json\', {**adapter.metadata, \'parameters\': params,\n                                      \'observed_compute_dtypes\': sorted(observed)})\n        if str(device).startswith(\'cuda\'):\n            torch.cuda.reset_peak_memory_stats(device)\n        rows = []\n        journal = out / \'predictions.partial.csv\'\n        if journal.exists():\n            journal.unlink()\n        for j, row in manifest.iterrows():\n            print(f"{adapter.metadata[\'model\']}: starting sample {j+1}/{len(manifest)} ({row.sample_id})", flush=True)\n            uav, block = load_pair(row)  # disk I/O excluded identically for all methods\n            for repeat in range(repeats):\n                seed = 2026 + int(j) * repeats + repeat\n                random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); cv2.setRNGSeed(seed)\n                sync()\n                start = time.perf_counter()\n                pred = adapter.predict(uav, block)\n                sync()\n                seconds = time.perf_counter() - start\n                x, y, valid, error = position_error(pred, row.gt_x, row.gt_y, meters_per_unit)\n                rows.append({\'sample_id\': row.sample_id, \'set\': row[\'set\'], \'repeat\': repeat,\n                             \'pred_x\': x, \'pred_y\': y, \'valid\': valid, \'error_m\': error,\n                             \'seconds\': seconds, \'inliers\': pred.get(\'inliers\'), \'n_angles\': pred.get(\'n_angles\')})\n                pd.DataFrame([rows[-1]]).to_csv(journal, mode=\'a\', header=not journal.exists(), index=False)\n                print(f"  repeat {repeat+1}/{repeats}: {seconds:.2f}s, angles={pred.get(\'n_angles\', \'-\')}", flush=True)\n        df = pd.DataFrame(rows)\n        df.to_csv(out / \'predictions.csv\', index=False)\n    # One fixed run per image for accuracy; all repeats for latency. No best-of-repeat selection.\n    first_run = df[df.repeat == 0]\n    summary = summarize(first_run, params, audit, adapter.metadata)\n    t = df.seconds.to_numpy()\n    summary.update(mean_ms=float(t.mean()*1000), p50_ms=float(np.median(t)*1000),\n                   p95_ms=float(np.percentile(t, 95)*1000), fps=float(1/t.mean()),\n                   latency_repeats=repeats, precision=\'fp32-audited\', batch_size=1,\n                   peak_allocated_MB=torch.cuda.max_memory_allocated(device)/2**20 if str(device).startswith(\'cuda\') else None)\n    pd.DataFrame([summary]).to_csv(out / \'summary.csv\', index=False)\n    groups = [summarize(g, params, audit, adapter.metadata) | {\'set\': name}\n              for name, g in first_run.groupby(\'set\')]\n    pd.DataFrame(groups).to_csv(out / \'per_set.csv\', index=False)\n    return summary\n', 'imatch/blocks.py': '"""파이프라인 ⑥: 후보 블록.\n\n좌표 규칙\n- 도시 지도 4096 px. 블록 (bx, by)는 지도의 [by*256, by*256+512) 행, [bx*256, bx*256+512) 열 (x가 열).\n- 블록은 256 px(2단위)씩 어긋나 있어 이웃과 절반씩 겹친다. 위치 범위는 0 <= bx, by < 15.\n- 후보 (bx+dx, by+dy) 기준 좌표 x\' 와 (bx, by) 기준 좌표 x 의 관계: x = x\' + 2 dx, y = y\' + 2 dy.\n"""\nN_BLOCK = 15\nUNITS_PER_BLOCK = 2  # 이웃 블록 사이 간격 (블록 단위)\n\n\ndef crop_block(city, bx, by):\n    """도시 지도에서 블록 (bx, by)를 512 px로 자른다."""\n    return city[by * 256:by * 256 + 512, bx * 256:bx * 256 + 512]\n\n\ndef neighbors(bx, by, radius):\n    """(dx, dy) 목록: 지도 범위 안의 (2 radius + 1)^2 후보."""\n    return [(dx, dy) for dx in range(-radius, radius + 1) for dy in range(-radius, radius + 1)\n            if 0 <= bx + dx < N_BLOCK and 0 <= by + dy < N_BLOCK]\n', 'imatch/geometry.py': '"""파이프라인 ④⑤: 대응점 검증(MAGSAC++)과 호모그래피 → 위치·heading.\n\n좌표 규칙 (Bearing-UAV와 같음)\n- 드론 영상 256 px, 지도 블록 512 px (2x2 타일). 블록 중심이 (0, 0), 1단위 = 128 px = 32 m.\n- 정답 변환: block_px = C + R(theta) (uav_px - 128),  R(t) = [[cos t, sin t], [-sin t, cos t]]\n"""\nimport numpy as np\n\nBLOCK_CENTER = 256.0\nPX_PER_UNIT = 128.0\nM_PER_UNIT = 32.0\n\nRANSAC_PX = 3.0  # MAGSAC++ 임계값(px). scripts/run.py --ransac-px 로 바꾼다\n\n\ndef set_ransac_px(px):\n    global RANSAC_PX\n    RANSAC_PX = float(px)\n\n\ndef rot(a):\n    t = np.deg2rad(a); c, s = np.cos(t), np.sin(t)\n    return np.array([[c, s], [-s, c]])\n\n\ndef turn(img, a):\n    """영상을 a도 돌린다 (반시계, cv2 규칙). 빈 모서리는 검은색."""\n    import cv2\n    h, w = img.shape[:2]\n    return cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), a, 1), (w, h), flags=cv2.INTER_LINEAR,\n                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)\n\n\ndef wrap(d):\n    """각도 차이의 절댓값 (0~180도)."""\n    return np.abs((np.asarray(d) + 180) % 360 - 180)\n\n\ndef magsac(p0, p1):\n    """대응점 → (p0, p1, 호모그래피 H, inlier 마스크). 점이 4개 미만이면 H = None."""\n    import cv2\n    if len(p0) < 4:\n        return p0, p1, None, np.zeros(len(p0), bool)\n    H, mask = cv2.findHomography(p0, p1, cv2.USAC_MAGSAC, RANSAC_PX, maxIters=2000, confidence=0.995)\n    return p0, p1, H, (mask[:, 0].astype(bool) if mask is not None else np.zeros(len(p0), bool))\n\n\ndef pose_from_homography(H, scale, angle, uav_center):\n    """돌린 드론 영상 → 블록 호모그래피에서 위치(x, y: 블록 단위)와 heading(도), 축척을 구한다.\n\n    위치: 드론 영상 중심을 블록으로 옮긴 점. heading: 중심에서의 국소 선형 변환 L의 회전 성분.\n    회전 탐색으로 영상을 angle만큼 돌렸으므로 L에 R(angle)을 곱해 원래 영상 기준으로 되돌린다.\n    """\n    S = np.diag([scale, scale, 1.])\n    Hb = np.linalg.inv(S) @ H @ S                                   # 축척 보정 (base px 기준)\n    h = Hb @ np.array([uav_center[0], uav_center[1], 1.]); c = h[:2] / h[2]  # 회전해도 중심은 그대로\n    L = (Hb[:2, :2] - np.outer(c, Hb[2, :2])) / h[2] @ rot(angle)   # 중심에서의 야코비안\n    return dict(px=float((c[0] - BLOCK_CENTER) / PX_PER_UNIT), py=float((c[1] - BLOCK_CENTER) / PX_PER_UNIT),\n                head=float(np.degrees(np.arctan2(L[0, 1], L[0, 0]))), scale_est=float(np.sqrt(abs(np.linalg.det(L)))))\n', 'imatch/matchers.py': '"""파이프라인 ②③: 특징 추출과 매칭. 모든 매처는 (드론 영상, 블록) → magsac() 결과를 돌려준다.\n\n새 매처를 실험하려면 __call__(im0, im1) 을 가진 클래스를 만들고 get_matcher()에 이름을 추가하면 된다.\n"""\nimport os\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import magsac\n\n# vismatch의 get_matcher에 없는 LoMa 변형은 직접 만든다 (\'loma\' = LoMa-B는 get_matcher로)\nLOMA_ARCH = {\'loma-b128\': \'LoMa-B128\', \'loma-l\': \'LoMa-L\', \'loma-g\': \'LoMa-G\'}\n\nclass VisMatch:\n    """vismatch 라이브러리의 공개 매처 (loma, romav2, sift-lightglue 등). GPU 권장.\n\n    LoMa 변형: \'loma\'(= LoMa-B, 주력), \'loma-b128\'(경량), \'loma-l\', \'loma-g\', \'loma-r\'.\n    """\n\n    def __init__(self, name, device=\'cuda\'):\n        import torch\n        self.name, self.torch = name, torch\n        if name in LOMA_ARCH:\n            from vismatch.im_models import loma\n            self.m = loma.LoMaMatcher(device, 2048, arch=LOMA_ARCH[name])\n        else:\n            from vismatch import get_matcher\n            self.m = get_matcher(name, device=device)\n        if name.startswith(\'loma\') and os.environ.get(\'LOMA_FP32\'):\n            # GTX 1650은 fp16에서 키포인트가 전부 NaN이 되어 오류 없이 짝 0개가 나온다. LOMA_FP32=1로 fp32를 쓴다.\n            # T4 결과(fp16)와 조금 다를 수 있으니 보고용 숫자는 T4 기본 설정으로 낸다\n            from vismatch.utils import force_float32\n            force_float32(self.m.matcher)\n        if name == \'romav2\':\n            # 기본 \'precise\' 설정은 T4(bf16 없음)에서 메모리 부족으로 멈췄다. 우리 영상은 256 / 512 px라\n            # \'base\' 설정(640 px, 단방향, fp32)을 쓴다.\n            from vismatch.im_models import romav2 as rv2\n            rv2._disable_autocast()\n            self.m.romav2_model.apply_setting(os.environ.get(\'ROMAV2_SETTING\', \'base\'))\n            self.name = f"romav2-{os.environ.get(\'ROMAV2_SETTING\', \'base\')}"\n\n    def __call__(self, im0, im1):\n        t = lambda im: self.torch.from_numpy(im).permute(2, 0, 1).float() / 255.\n        r = self.m(t(im0), t(im1))\n        p0, p1 = r[\'matched_kpts0\'], r[\'matched_kpts1\']\n        # vismatch는 inlier 점만 주고 마스크는 안 주므로 같은 점으로 MAGSAC++를 다시 돌린다\n        return magsac(np.float32(p0).reshape(-1, 2), np.float32(p1).reshape(-1, 2))\n\n\nclass CvSift:\n    """GPU 없이 도는 기준선: OpenCV SIFT + Lowe 비율 검사 + MAGSAC++. 설치·동작 확인용."""\n    name = \'cv-sift\'\n\n    def __init__(self):\n        self.sift = cv2.SIFT_create(4096); self.bf = cv2.BFMatcher()\n\n    def __call__(self, im0, im1):\n        k0, d0 = self.sift.detectAndCompute(cv2.cvtColor(im0, cv2.COLOR_RGB2GRAY), None)\n        k1, d1 = self.sift.detectAndCompute(cv2.cvtColor(im1, cv2.COLOR_RGB2GRAY), None)\n        if d0 is None or d1 is None or len(k0) < 2 or len(k1) < 2:\n            return np.empty((0, 2)), np.empty((0, 2)), None, np.zeros(0, bool)\n        mm = [m for m, n in self.bf.knnMatch(d0, d1, k=2) if m.distance < 0.8 * n.distance]\n        p0 = np.float32([k0[m.queryIdx].pt for m in mm]).reshape(-1, 2)\n        p1 = np.float32([k1[m.trainIdx].pt for m in mm]).reshape(-1, 2)\n        return magsac(p0, p1)\n\n\nclass VggMNN:\n    """비교 실험 \'VGG16 백본은 그대로 두면?\' (docs/02_실험기록.md 7장). 원 Bearing-UAV 체크포인트의 고정\n    VGG16 conv4_3(B2가 상관을 계산하는 그 특징)을 점 단위 기술자로 쓴다.\n\n    특징 맵의 칸(stride 8, 중심 8 i + 4)이 점이다. 블록 평균을 뺀 코사인 유사도로 상호 최근접 짝을 짓고,\n    이후는 다른 매처와 같은 MAGSAC++ / 호모그래피를 쓴다. 회전으로 생긴 검은 모서리의 칸은 뺀다.\n    capstone1 저장소와 체크포인트가 필요하다:\n      CAPSTONE1_DIR=<capstone1 경로> VGG_CKPT=Bearing_UAV/cross_view python scripts/run.py --matchers vgg-mnn --device cpu\n    결과: Singapore 44%(B2 69%, 100샘플) → 원 모델 특징으로는 점 단위 매칭이 안 된다.\n    """\n    name = \'vgg-mnn\'\n\n    def __init__(self):\n        import sys\n        import torch\n        root = os.environ[\'CAPSTONE1_DIR\']\n        sys.path.insert(0, root)\n        from diag.diag_modules import DEV, load_model, to_tensor\n        self.torch, self.tt, self.dev = torch, to_tensor, DEV\n        self.bb = load_model(os.path.join(root, os.environ.get(\'VGG_CKPT\', \'Bearing_UAV/cross_view\'))).backbone\n\n    def _feat(self, im):\n        with self.torch.no_grad():\n            f = self.bb(self.tt(im).unsqueeze(0).to(self.dev))[0]          # [C, h, w]\n        C, h, w = f.shape; st = im.shape[0] / h\n        yy, xx = np.mgrid[0:h, 0:w]\n        pts = np.stack([xx.ravel() * st + st / 2, yy.ravel() * st + st / 2], 1).astype(np.float32)\n        return f.reshape(C, -1).T, pts\n\n    def __call__(self, im0, im1):\n        F = self.torch.nn.functional\n        d0, p0 = self._feat(im0); d1, p1 = self._feat(im1)\n        valid = im0[np.clip(p0[:, 1].astype(int), 0, im0.shape[0] - 1), np.clip(p0[:, 0].astype(int), 0, im0.shape[1] - 1)].sum(1) > 0\n        d0, p0 = d0[self.torch.from_numpy(valid)], p0[valid]\n        mu = d1.mean(0, keepdim=True)\n        d0, d1 = F.normalize(d0 - mu, dim=1), F.normalize(d1 - mu, dim=1)\n        S = d0 @ d1.T\n        j = S.argmax(1); i = S.argmax(0)\n        mutual = (i[j] == self.torch.arange(len(j), device=j.device)).cpu().numpy()\n        j = j.cpu().numpy()\n        return magsac(p0[mutual], p1[j[mutual]])\n\n\ndef get_matcher(name, device=\'cuda\'):\n    if name == \'cv-sift\':\n        return CvSift()\n    if name == \'vgg-mnn\':\n        return VggMNN()\n    return VisMatch(name, device)\n', 'imatch/metrics.py': '"""팀 공통 평가 규칙 (2026-10-15 확정). 모든 표의 숫자는 이 규칙으로 계산한다.\n\n- 추정은 2x2 블록 안으로 clip 한다. 추정이 없으면 블록 중심 (0, 0)을 답으로 낸다.\n  (B2·CorrMatchNet은 원래 블록 안만 답할 수 있으므로 같은 조건을 맞춘 것)\n- 실패 = 추정 없음 또는 블록 밖으로 0.1단위(3.2 m) 넘게 나감.\n- 성공 = 위치 오차 10 m 이내.\n- 비교는 같은 샘플끼리 짝지어 McNemar 정확 검정 (나만 성공 / 상대만 성공).\n"""\nimport numpy as np\nfrom scipy.stats import binomtest\n\nfrom .blocks import UNITS_PER_BLOCK\nfrom .geometry import M_PER_UNIT\n\n\ndef errors(pred_x, pred_y, gt_x, gt_y, lim=1.0):\n    """(위치 오차 m, 실패 여부) 배열. lim = clip 범위 (블록 단위)."""\n    px, py = np.asarray(pred_x, float), np.asarray(pred_y, float)\n    found = np.isfinite(px) & np.isfinite(py)\n    px, py = np.nan_to_num(px), np.nan_to_num(py)\n    fail = ~found | (np.abs(px) > lim + 0.1) | (np.abs(py) > lim + 0.1)\n    e = np.hypot(np.clip(px, -lim, lim) - np.asarray(gt_x), np.clip(py, -lim, lim) - np.asarray(gt_y)) * M_PER_UNIT\n    return e, fail\n\n\ndef errors_chosen_block(pred_x, pred_y, dx, dy, gt_x, gt_y):\n    """후보 블록 결과용 errors. pred는 고른 블록(정답 블록 + (dx, dy)) 기준이고, clip은 그 블록의 영상 범위(±2)로 한다.\n\n    errors의 기본 ±1은 정답 블록일 때만 맞다. 정답 블록 좌표로 옮겨서 ±1로 clip하면 이웃 블록을 고른 추정이 정답\n    쪽으로 끌려와 오차가 작게 나오고, 고른 블록 기준 ±1로 clip하면 이웃 블록의 맞는 추정(예: -1.45)이 틀어진다.\n    이 함수의 숫자는 errors로 잰 표와 바로 비교하면 안 된다. 같은 후보 결과끼리만 비교한다.\n    """\n    dx, dy = np.nan_to_num(np.asarray(dx, float)), np.nan_to_num(np.asarray(dy, float))\n    return errors(pred_x, pred_y, np.asarray(gt_x) - UNITS_PER_BLOCK * dx, np.asarray(gt_y) - UNITS_PER_BLOCK * dy, lim=2.0)\n\n\ndef mcnemar(ok_mine, ok_other):\n    """\'나만 성공 / 상대만 성공, p\' 문자열."""\n    a, b = int((ok_mine & ~ok_other).sum()), int((~ok_mine & ok_other).sum())\n    return f"{a} / {b}, p = {binomtest(a, a + b).pvalue:.2g}" if a + b else "0 / 0"\n', 'imatch/nav.py': '"""닫힌 루프 1 km 항법. capstone1 diag/diag_phase0.py의 make_traj, Cropper, nav_run을 옮긴 것이다.\n\n바뀐 점은 드론 영상을 frames(x, y, heading) → (영상, 영상의 실제 x, y)로 받는 것뿐이다.\n- Crop2D: 기존 T5와 같다. 지도를 잘라 드론 영상으로 쓴다 (2D는 SIFT로도 풀리는 시험이라 동작 확인용)\n- Nearest3D: 궤적 지점에서 가장 가까운 3D 렌더를 쓴다. 연속 프레임이 없어서 만든 근사다.\n  렌더는 도시당 22,500장이고 궤적까지 거리 중앙값 3.0 m, 최대 약 11 m. heading은 렌더마다 무작위라\n  비행 방향과 맞지 않는다. 오차는 렌더의 실제 위치로 잰다.\n"""\nimport hashlib\nimport os\nimport re\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial import cKDTree\n\nfrom .blocks import N_BLOCK, crop_block, neighbors\nfrom .geometry import PX_PER_UNIT\nfrom .pipeline import localize\n\nPX = 4.0  # px per metre (0.25 m/px)\nW = 384\nMETA_3D = \'Bearing_UAV_90K/c4m_254k_96bc_b15_s100_v3d/metadata/metadata.csv\'\n\n\ndef make_traj(n_traj, dist_m, seed, n_block=15):\n    rng = np.random.default_rng(seed)\n    lo, hi = 300, 256 * (n_block + 1) - 300\n    starts, goals = [], []\n    while len(starts) < n_traj:\n        s = rng.uniform(lo, hi, 2); a = rng.uniform(0, 2 * np.pi)\n        g = s + dist_m * PX * np.array([np.cos(a), np.sin(a)])\n        if (g > lo).all() and (g < hi).all():\n            starts.append(s); goals.append(g)\n    return np.array(starts), np.array(goals)\n\n\nclass Crop2D:\n    """지도에서 드론 영상을 자른다 (capstone1 Cropper와 같은 결과)."""\n    def __init__(self, city):\n        self.pad = cv2.copyMakeBorder(city, W, W, W, W, cv2.BORDER_REFLECT)\n\n    def __call__(self, cx, cy, theta):\n        x0, y0 = int(cx) - W, int(cy) - W\n        win = self.pad[y0 + W:y0 + 3 * W, x0 + W:x0 + 3 * W]\n        lx, ly = cx - x0, cy - y0\n        R = cv2.getRotationMatrix2D((lx, ly), -theta, 1)\n        rot = cv2.warpAffine(win, R, (win.shape[1], win.shape[0]), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)\n        c_x, c_y = int(lx), int(ly)\n        return rot[c_y - 128:c_y + 128, c_x - 128:c_x + 128], cx, cy\n\n\nclass Nearest3D:\n    """가장 가까운 3D 렌더를 돌려준다. capstone1의 Bearing_UAV_90K가 필요하다."""\n    def __init__(self, capstone1, st):\n        ho = re.search(r\'(\\d+bc)\', st).group(1)\n        df = pd.read_csv(os.path.join(capstone1, META_3D), usecols=[\'block_x\', \'block_y\', \'x_norm\', \'y_norm\', \'target_path\'])\n        df = df[df.target_path.str.contains(f\'_{ho}_\')]\n        self.xy = np.c_[df.block_x * 256 + 256 + df.x_norm * PX_PER_UNIT, df.block_y * 256 + 256 + df.y_norm * PX_PER_UNIT]\n        self.paths = [os.path.join(capstone1, p) for p in df.target_path]\n        self.tree = cKDTree(self.xy)\n\n    def __call__(self, cx, cy, theta):\n        i = self.tree.query((cx, cy))[1]\n        return cv2.cvtColor(cv2.imread(self.paths[i]), cv2.COLOR_BGR2RGB), self.xy[i, 0], self.xy[i, 1]\n\n\nclass MatchLoc:\n    """localize를 nav_run의 loc(crops, headings, blocks) 형식으로 감싼다. 위치는 블록 단위 (1 = 128 px).\n\n    추정이 없으면 블록 중심을 답한다 (imatch/metrics.py 규칙과 같음). 나침반은 쓰지 않으므로 headings는 무시한다.\n    걸음별 기록은 self.last에 남고, nav_run의 log에 들어간다.\n    같은 (영상, 블록)은 다시 매칭하지 않는다 (self.cache). R0~R4를 한 프로세스에서 돌리면 겹치는 걸음을 아낀다.\n    stop_inl > 0이면 각도 탐색을 inlier가 stop_inl 이상인 첫 각도에서 멈춘다 (localize 조기 종료).\n    """\n    def __init__(self, matcher, city, scale=1.0, angles=(0.0,), stop_inl=0):\n        self.matcher, self.city, self.scale, self.angles, self.last, self.cache = matcher, city, scale, angles, [], {}\n        self.stop_inl = stop_inl\n\n    def __call__(self, crops, headings, blocks):\n        out, self.last = [], []\n        for crop, (bx, by) in zip(crops, blocks):\n            xy, rec = self.one(crop, bx, by)\n            out.append(xy); self.last.append(rec)\n        return np.array(out)\n\n    def one(self, crop, bx, by):\n        """블록 하나: (위치(블록 단위), 기록). self.last는 건드리지 않는다 (Research가 nav_run 걸음 중간에 부른다)."""\n        key = (hashlib.md5(crop.tobytes()).hexdigest(), bx, by)\n        if key not in self.cache:\n            recs, best = localize(self.matcher, crop, crop_block(self.city, bx, by), (0.0, 0.0, 0.0), self.scale, self.angles,\n                                  stop_inl=self.stop_inl)\n            self.cache[key] = ((best[\'px\'], best[\'py\']) if best else (0.0, 0.0),\n                               dict(n_inl=best[\'n_inl\'] if best else 0, found=int(best is not None), sec=sum(r[\'sec\'] for r in recs),\n                                    n_ang=len(recs)))\n        return self.cache[key]\n\n\nclass NoMatch:\n    """매칭하지 않는 기준선. 항상 블록 중심을 답하고 inlier 0을 돌려준다.\n\n    reject_inl = 0이면 "항상 중앙", reject_inl > 0이면 모든 걸음이 "모름"이라 추측 항법만으로 난다.\n    """\n    def __call__(self, crops, headings, blocks):\n        self.last = [dict(n_inl=0, found=0, sec=0.0) for _ in crops]\n        return np.zeros((len(crops), 2))\n\n\nclass Research:\n    """R3·R4: "모름"이면 추측 위치 주변 블록을 다시 매칭한다. nav_run의 recover로 넘긴다.\n\n    R3: 추측 위치의 블록과 십자 이웃 4개 (⑥ 블록 선택의 후보 5개, experiments/2026-11-06_block_select).\n    R4 (wide_after > 0): 연속 "모름"이 wide_after걸음 이상이면 3×3 9개.\n    통과 = 그 블록 기준 추정이 ±inside 안 (inside 규칙)이고 inlier >= min_inl. 여러 개면 추측 위치에 가장 가까운 것, 없으면 None.\n    gate_m > 0이면 십자 재탐색의 답도 추측 위치에서 gate_m m 안이어야 한다 (R2와 같은 움직임 검사).\n    넓게 찾기는 추측 위치가 이미 밀렸다고 보고 움직임 검사를 하지 않는다. 대신 wide_inl > 0이면 넓게 찾기에서만\n    inlier >= wide_inl을 요구한다 (R4 학습 도시에서 틀린 3×3 답은 inlier 중앙값 16, 맞은 답은 26).\n    추측 위치의 블록은 이번 걸음에 이미 매칭했으므로 캐시에서 나온다. 부를 때마다 self.log에 한 줄 남긴다\n    (rec_sec = 이번 걸음에 더 든 매칭 시간, 추측 위치의 블록 제외).\n    """\n    def __init__(self, match, min_inl=15, gate_m=0., wide_after=0, inside=1.1, wide_inl=0):\n        self.match, self.min_inl, self.gate_m, self.wide_after, self.inside = match, min_inl, gate_m, wide_after, inside\n        self.wide_inl = wide_inl\n        self.log = []\n\n    def __call__(self, i, crop, guess, streak):\n        wide = self.wide_after > 0 and streak >= self.wide_after\n        min_inl = max(self.min_inl, self.wide_inl) if wide else self.min_inl\n        bx = int(np.clip(np.round((guess[0] - 256) / 256), 0, N_BLOCK - 1))\n        by = int(np.clip(np.round((guess[1] - 256) / 256), 0, N_BLOCK - 1))\n        cands = [(dx, dy) for dx, dy in neighbors(bx, by, 1) if wide or abs(dx) + abs(dy) <= 1]\n        best, best_d, n_pass, sec = None, np.inf, 0, 0.\n        for dx, dy in cands:\n            (x, y), rec = self.match.one(crop, bx + dx, by + dy)\n            sec += rec[\'sec\'] if (dx, dy) != (0, 0) else 0.\n            if not rec[\'found\'] or rec[\'n_inl\'] < min_inl or max(abs(x), abs(y)) > self.inside:\n                continue\n            p = np.array([(bx + dx) * 256 + 256, (by + dy) * 256 + 256]) + np.array([x, y]) * PX_PER_UNIT\n            d = np.linalg.norm(p - guess) / PX\n            if self.gate_m > 0 and not wide and d > self.gate_m:\n                continue\n            n_pass += 1\n            if d < best_d:\n                best, best_d, pick = p, d, (bx + dx, by + dy, rec[\'n_inl\'])\n        self.log.append(dict(traj=int(i), rec_wide=int(wide), rec_cand=len(cands), rec_pass=n_pass, rec_sec=sec,\n                             **(dict(zip((\'rec_bx\', \'rec_by\', \'rec_n_inl\'), pick)) if best is not None else {})))\n        return best\n\n\nODO = {\'none\': None, \'vio\': dict(bias_deg=3.0, head_deg=2.0, scale=0.05)}\n"""이동 오차 (추측 항법이 공짜로 맞지 않게).\n\n시뮬레이션의 드론은 명령한 대로 정확히 움직이므로, 오차가 없으면 영상을 안 쓰고 추측 항법만 해도 100% 도착한다.\n\'vio\': 궤적마다 heading 치우침 N(0, 3°), 걸음마다 heading N(0, 2°), 걸음 길이 N(1, 5%).\n1 km에서 heading 3° 치우침만으로 옆으로 약 52 m 밀린다. 드론은 이 오차를 모르고 명령한 이동량으로 추측한다.\n"""\n\n\ndef nav_run(loc, frames, starts, goals, decode_px=128, oracle_block=False, step_m=15., arrive_m=20., n_block=15, log=None,\n            odo=None, seed=0, reject_inl=0, gate_m=0., recover=None):\n    """(성공 여부, 종료 이유, 궤적별 걸음 오차 중앙값 m). log: 걸음마다 기록을 덧붙일 목록.\n\n    복구 (docs/04 R0~R4). 기본값이면 R0이고 예전 결과와 같다.\n    odo: ODO의 값. 실제 이동에 오차를 넣는다. 같은 seed면 모든 R에서 같은 오차가 나온다.\n    reject_inl: 고른 답의 inlier가 이보다 작으면 "모름" (R1, 15).\n    gate_m: 추정이 추측 위치(직전 위치 + 명령 이동량)에서 이보다 멀면 "모름" (R2).\n    recover: "모름"일 때 부르는 함수 recover(i, crop, guess_px, streak) → 지도 px 위치 또는 None (R3, R4 자리).\n             None을 돌려주거나 recover가 없으면 추측 위치를 쓴다. streak = 이 궤적의 연속 "모름" 걸음 수 (이번 걸음 포함).\n    """\n    n = len(starts); dist_m = np.linalg.norm(goals[0] - starts[0]) / PX\n    r = starts.copy(); nom = r.copy()\n    alive = np.ones(n, bool); success = np.zeros(n, bool); reason = np.array([\'timeout\'] * n, dtype=object)\n    heading = np.full(n, 90.0); err = [[] for _ in range(n)]; streak = np.zeros(n, int)\n    T = int(3 * dist_m / step_m)\n    rng = np.random.default_rng(seed + 1)\n    bias = rng.normal(0, odo[\'bias_deg\'], n) if odo else np.zeros(n)\n    dh = rng.normal(0, odo[\'head_deg\'], (n, T)) if odo else np.zeros((n, T))\n    ds = rng.normal(0, odo[\'scale\'], (n, T)) if odo else np.zeros((n, T))\n    for t in range(T):\n        idx = np.where(alive)[0]\n        if len(idx) == 0:\n            break\n        blocks, crops, cen, real = [], [], [], []\n        for i in idx:\n            ref = r[i] if oracle_block else nom[i]\n            bx = int(np.clip(np.round((ref[0] - 256) / 256), 0, n_block - 1))\n            by = int(np.clip(np.round((ref[1] - 256) / 256), 0, n_block - 1))\n            blocks.append((bx, by)); cen.append(np.array([bx * 256 + 256, by * 256 + 256]))\n            crop, fx, fy = frames(r[i, 0], r[i, 1], heading[i])\n            crops.append(crop); real.append(np.array([fx, fy]))\n        pos = loc(crops, heading[idx], blocks)\n        last = getattr(loc, \'last\', None) or [{} for _ in idx]\n        for j, i in enumerate(idx):\n            raw = cen[j] + pos[j] * decode_px\n            guess = nom[i].copy()  # 드론이 아는 것만으로 추측한 이번 위치 (복사: 아래에서 nom[i]를 고친다)\n            n_inl = last[j].get(\'n_inl\', np.inf)\n            unknown = (reject_inl > 0 and n_inl < reject_inl) or (gate_m > 0 and np.linalg.norm(raw - guess) / PX > gate_m)\n            streak[i] = streak[i] + 1 if unknown else 0\n            p, how = raw, \'match\'\n            if unknown:\n                q = recover(i, crops[j], guess, streak[i]) if recover else None\n                p, how = (np.asarray(q, float), \'recover\') if q is not None else (guess, \'guess\')\n                if q is not None: streak[i] = 0\n            err[i].append(float(np.linalg.norm(p - real[j]) / PX))\n            if log is not None:\n                log.append(dict(traj=int(i), step=t, real_x=float(real[j][0]), real_y=float(real[j][1]), est_x=float(p[0]),\n                                est_y=float(p[1]), block_x=blocks[j][0], block_y=blocks[j][1], err_m=err[i][-1],\n                                outside_block=int(np.abs(real[j] - cen[j]).max() > 128),\n                                raw_err_m=float(np.linalg.norm(raw - real[j]) / PX), unknown=int(unknown), used=how,\n                                **last[j]))\n            d = goals[i] - p; u = d / max(np.linalg.norm(d), 1e-6)\n            heading[i] = np.degrees(np.arctan2(-u[1], u[0]))\n            a = np.radians(bias[i] + dh[i, t]); c, s_ = np.cos(a), np.sin(a)\n            move = np.array([c * u[0] - s_ * u[1], s_ * u[0] + c * u[1]]) * (1 + ds[i, t])\n            r[i] = r[i] + move * step_m * PX; nom[i] = p + u * step_m * PX\n            real_d = np.linalg.norm(r[i] - goals[i]) / PX; pred_d = np.linalg.norm(p - goals[i]) / PX\n            if real_d < arrive_m or pred_d < arrive_m:\n                alive[i] = False; success[i] = real_d < 2 * arrive_m\n                reason[i] = \'arrived_ok\' if success[i] else \'arrived_pred_only\'\n            elif (r[i] < 200).any() or (r[i] > 256 * (n_block + 1) - 200).any():\n                alive[i] = False; reason[i] = \'out_of_map\'\n    return success, reason, [float(np.median(e)) for e in err if e]\n', 'imatch/pipeline.py': '"""파이프라인 ①~⑤를 한 샘플에 적용한다.\n\n① 회전 탐색: 드론 영상을 angles의 각 각도로 돌린다 (angles=[0]이면 탐색 없음, 원 Bearing-UAV와 같은 조건)\n②③ 매칭: matcher(돌린 드론 영상, 블록)\n④ 검증: MAGSAC++ inlier\n⑤ 위치·heading: 호모그래피 → pose_from_homography\n→ inlier가 가장 많은 각도의 결과를 고른다.\n   stop_inl > 0이면 각도를 차례로 해 보다가 inlier가 stop_inl 이상인 각도가 나오면 거기서 멈춘다 (조기 종료, 빠르게).\n"""\nimport time\n\nimport cv2\nimport numpy as np\n\nfrom .geometry import pose_from_homography, rot, turn\n\n\ndef localize(matcher, uav, block, gt, scale=1.0, angles=(0.0,), px_tol=8.0, stop_inl=0):\n    """각도별 기록 목록과 고른 결과(best, 없으면 None)를 돌려준다.\n\n    gt = (x_norm, y_norm, theta): 위치 추정에는 쓰지 않고, 대응점이 맞는지(good_*) 채점하는 데만 쓴다.\n    """\n    hu, wu = uav.shape[:2]; cu = np.array([wu / 2, hu / 2])\n    U = cv2.resize(uav, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else uav\n    B = cv2.resize(block, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC) if scale != 1 else block\n    xn, yn, th = gt; C = np.array([256 + xn * 128, 256 + yn * 128]); Rg = rot(th)\n    recs = []\n    for a in angles:\n        t0 = time.time()\n        p0, p1, H, inl = matcher(turn(U, a), B)\n        dt = time.time() - t0\n        p0 = np.asarray(p0, np.float64) / scale; p1 = np.asarray(p1, np.float64) / scale\n        # 원래 드론 영상 좌표로 되돌림: p_turned = cu + R(a) (p - cu)  ->  p = cu + R(a)^T (p_turned - cu)\n        po = (p0 - cu) @ rot(a) + cu if len(p0) else p0\n        good = np.linalg.norm(C + (po - cu) @ Rg.T - p1, axis=1) < px_tol if len(p0) else np.zeros(0, bool)\n        r = dict(angle=a, n_match=int(len(p0)), n_inl=int(inl.sum()), sec=dt,\n                 good_match=float(good.mean()) if len(good) else np.nan,\n                 good_inl=float(good[inl].mean()) if inl.sum() else np.nan)\n        if H is not None and inl.sum() >= 4:\n            r.update(pose_from_homography(H, scale, a, cu))\n        recs.append(r)\n        if stop_inl > 0 and \'px\' in r and r[\'n_inl\'] >= stop_inl:\n            break\n    ok = [r for r in recs if \'px\' in r]\n    best = max(ok, key=lambda r: r[\'n_inl\']) if ok else None\n    return recs, best\n', 'imatch/__init__.py': '"""Trone 이미지 매칭 파이프라인: 드론 영상 + 지도 블록 → 위치·heading (학습 없음).\n\n  geometry.py  ④⑤ MAGSAC++ 검증, 호모그래피 → 위치·heading\n  matchers.py  ②③ 특징 추출·매칭 (LoMa, RoMa v2, SIFT ...)\n  pipeline.py  ①~⑤ 한 샘플 처리 (회전 탐색 포함)\n  blocks.py    ⑥ 후보 블록 자르기, 이웃 블록 좌표\n  nav.py       1 km 닫힌 루프 항법 (2D 크롭, 3D 렌더 입력)\n  metrics.py   팀 공통 평가 규칙\n"""\ndef __getattr__(name):\n    # CPU evaluation should not require GPU/image-processing dependencies.\n    if name == \'get_matcher\':\n        from .matchers import get_matcher\n        return get_matcher\n    if name == \'localize\':\n        from .pipeline import localize\n        return localize\n    raise AttributeError(name)\n', 'retrieval_worker.py': '"""Isolated official Sample4Geo/MEAN inference; launched by additive notebook cells."""\nimport argparse\nimport ast\nimport contextlib\nimport hashlib\nimport importlib\nimport json\nimport math\nimport sys\nimport time\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024**2), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef distance_m(q, g, mode):\n    if mode == \'wgs84\':\n        lat1, lat2 = math.radians(q[\'lat\']), math.radians(g[\'lat\'])\n        dlat = lat2 - lat1\n        dlon = math.radians(g[\'lon\'] - q[\'lon\'])\n        a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2\n        return 6371008.8 * 2 * math.asin(math.sqrt(min(1., max(0., a))))\n    if q[\'frame_id\'] != g[\'frame_id\']:\n        raise ValueError(\'Query and retrieved Gallery have different metric coordinate frames\')\n    return math.hypot(q[\'east_m\']-g[\'east_m\'], q[\'north_m\']-g[\'north_m\'])\n\n\ndef load_coordinates(qcsv, gallery, queries, approved, provenance):\n    if not approved:\n        return None, None, \'Coordinate provenance not approved; location accuracy N/A\'\n    if not provenance.strip() or not qcsv:\n        return None, None, \'Missing coordinate provenance or Query coordinate CSV\'\n    q = pd.read_csv(qcsv, dtype={\'sample_id\': str})\n    if q.sample_id.duplicated().any() or set(q.sample_id) != set(queries.sample_id):\n        raise ValueError(\'Query coordinates must match all original sample IDs exactly\')\n    q = q.set_index(\'sample_id\').loc[queries.sample_id].reset_index()\n    if {\'lat\',\'lon\'}.issubset(q.columns) and {\'lat\',\'lon\'}.issubset(gallery.columns):\n        cols, mode = [\'lat\',\'lon\'], \'wgs84\'\n        for t in (q, gallery):\n            if not np.isfinite(t[cols].to_numpy(float)).all() or (t.lat.abs()>90).any() or (t.lon.abs()>180).any():\n                raise ValueError(\'Invalid WGS84 coordinates\')\n    elif {\'east_m\',\'north_m\',\'frame_id\'}.issubset(q.columns) and {\'east_m\',\'north_m\',\'frame_id\'}.issubset(gallery.columns):\n        cols, mode = [\'east_m\',\'north_m\'], \'metric\'\n        for t in (q, gallery):\n            if not np.isfinite(t[cols].to_numpy(float)).all() or t.frame_id.isna().any():\n                raise ValueError(\'Invalid metric coordinates/frame IDs\')\n        # Search must not use query GT to choose a coordinate frame or restrict candidates.\n        if len(set(q.frame_id) | set(gallery.frame_id)) != 1:\n            raise ValueError(\'Use a shared metric CRS for the entire bank, or WGS84. No GT-based filtering.\')\n    else:\n        return None, None, \'No shared WGS84 or metric coordinates; local gt_x/gt_y cannot locate other blocks\'\n    return q, mode, \'\'\n\n\ndef load_state(path):\n    state = torch.load(path, map_location=\'cpu\', weights_only=True)\n    for key in (\'state_dict\', \'model_state_dict\', \'model\'):\n        if isinstance(state, dict) and key in state and isinstance(state[key], dict):\n            state = state[key]\n            break\n    if not isinstance(state, dict) or not state or not all(isinstance(v, torch.Tensor) for v in state.values()):\n        raise ValueError(\'Expected an official tensor state_dict checkpoint\')\n    keys = list(state)\n    if all(k.startswith(\'module.\') for k in keys):\n        state = {k[7:]: v for k,v in state.items()}\n    return state\n\n\ndef official_transform(repo, model):\n    # Execute only the upstream validation assignment, avoiding training augmentation\n    # construction incompatible with modern albumentations. No interpolation/normalization changes.\n    import albumentations as A\n    from albumentations.pytorch import ToTensorV2\n    tree = ast.parse((repo/\'sample4geo/dataset/university.py\').read_text(encoding=\'utf-8-sig\'))\n    fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'get_transforms\')\n    assignment = next(n for n in fn.body if isinstance(n, ast.Assign) and\n                      any(isinstance(t, ast.Name) and t.id == \'val_transforms\' for t in n.targets))\n    cfg = model.get_config()\n    ns = dict(A=A, cv2=cv2, ToTensorV2=ToTensorV2, img_size=(384,384), mean=cfg[\'mean\'], std=cfg[\'std\'])\n    exec(compile(ast.Module(body=[assignment], type_ignores=[]), \'<official_validation_transform>\', \'exec\'), ns)\n    return ns[\'val_transforms\']\n\n\ndef load_model(name, repo, checkpoint):\n    sys.path.insert(0, str(repo))\n    state = load_state(checkpoint)  # missing/invalid weights fail before construction\n    if name == \'Sample4Geo\':\n        from sample4geo.model import TimmModel\n        model = TimmModel(\'convnext_base.fb_in22k_ft_in1k_384\', pretrained=False, img_size=384)\n    else:\n        from sample4geo.hand_convnext.model import make_model\n        # Official registration replaces timm convnext_tiny with the repository implementation.\n        # Suppress redundant ImageNet initialization downloads only; full strict checkpoint load follows.\n        module = importlib.import_module(\'sample4geo.hand_convnext.ConvNext.make_model\')\n        original = module.create_model\n        def no_download(*args, **kwargs):\n            kwargs[\'pretrained\'] = False\n            return original(*args, **kwargs)\n        module.create_model = no_download\n        try:\n            model = make_model(SimpleNamespace(views=2, nclasses=701, block=2, triplet_loss=.3, resnet=False))\n        finally:\n            module.create_model = original\n    model.load_state_dict(state, strict=True)\n    del state\n    model = model.float().cuda().eval()\n    transform = official_transform(repo, model)\n    return model, transform\n\n\ndef rgb(path):\n    im = cv2.imread(str(path))\n    if im is None:\n        raise FileNotFoundError(path)\n    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)\n\n\ndef sync():\n    torch.cuda.synchronize()\n\n\ndef feature(model, transform, image, name):\n    x = transform(image=image)[\'image\'].unsqueeze(0).to(\'cuda\', dtype=torch.float32)\n    sync()\n    start = time.perf_counter()\n    with torch.autocast(\'cuda\', enabled=False):\n        output = model(x)\n    sync()\n    forward_ms = (time.perf_counter()-start)*1000\n    if name == \'MEAN\':\n        output = output[-2]  # official trainer.predict: global feature, NOT part feature map\n    if output.ndim != 2 or output.shape[0] != 1 or output.dtype != torch.float32 or not torch.isfinite(output).all():\n        raise ValueError(f\'Invalid official embedding: {output.shape}, {output.dtype}\')\n    return F.normalize(output, dim=-1), forward_ms\n\n\ndef evaluate(config):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU required; no CPU substitution\')\n    torch.cuda.set_device(config[\'gpu_index\'])\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.manual_seed(2026)\n    name = config[\'name\']\n    out = Path(config[\'output\'])\n    out.mkdir(parents=True, exist_ok=True)\n    result_path = out/\'summary.json\'\n    config_signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode() + Path(__file__).read_bytes()).hexdigest()\n    if result_path.exists():\n        saved=json.loads(result_path.read_text())\n        if (saved.get(\'config_signature\') != config_signature or saved.get(\'checkpoint_sha256\') != digest(config[\'checkpoint\']) or\n            saved.get(\'queries_sha256\') != digest(config[\'queries\']) or\n            saved.get(\'gallery_sha256\') != digest(config[\'gallery\'])):\n            raise ValueError(\'Existing result belongs to different weights/data. Start a new addon run; preserved old result.\')\n        print(\'Preserved completed result:\', result_path, flush=True)\n        return saved\n    queries = pd.read_csv(config[\'queries\'], dtype={\'sample_id\': str})\n    gallery = pd.read_csv(config[\'gallery\'], dtype={\'gallery_id\': str})\n    if queries.empty or queries.sample_id.duplicated().any():\n        raise ValueError(\'Expected nonempty unique queries\')\n    if gallery.empty or gallery.gallery_id.duplicated().any():\n        raise ValueError(\'Gallery must have unique IDs and at least one image\')\n    qgeo, mode, reason = load_coordinates(config[\'query_geo\'], gallery, queries,\n                                         config[\'coordinates_verified\'], config[\'coordinate_provenance\'])\n    smoke = config[\'smoke\']\n    if smoke:\n        queries = queries.head(3).copy()\n        gallery = gallery.head(3).copy()\n        qgeo, mode, reason = None, None, \'Smoke test only: three Queries/Gallery images; accuracy N/A\'\n    model, transform = load_model(name, Path(config[\'repo\']), config[\'checkpoint\'])\n    params = sum(p.numel() for p in model.parameters())\n    observed, hooks = set(), []\n    def capture(module, args, output):\n        if isinstance(output, torch.Tensor):\n            observed.add(str(output.dtype))\n    for module in model.modules():\n        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):\n            hooks.append(module.register_forward_hook(capture))\n    with torch.inference_mode():\n        first = rgb(queries.iloc[0].uav_path)\n        feature(model, transform, first, name)\n        for hook in hooks:\n            hook.remove()\n        if observed != {\'torch.float32\'}:\n            raise RuntimeError(f\'Compute dtype audit failed: {observed}\')\n        for _ in range(config[\'warmup\']):\n            feature(model, transform, first, name)\n        torch.cuda.reset_peak_memory_stats()\n        gallery_rows, embeddings = [], []\n        offline_start = time.perf_counter()\n        for i, row in gallery.iterrows():\n            image = rgb(row.image_path)  # gallery wall time includes disk I/O, separately labeled\n            sync(); start = time.perf_counter()\n            emb, forward_ms = feature(model, transform, image, name)\n            sync(); sec = time.perf_counter()-start\n            embeddings.append(emb.cpu())\n            gallery_rows.append(dict(gallery_id=row.gallery_id, seconds=sec, forward_ms=forward_ms))\n            if (i+1)%100 == 0:\n                print(f\'{name}: Gallery {i+1}/{len(gallery)}\', flush=True)\n        bank = torch.cat(embeddings).to(\'cuda\')\n        sync()\n        offline_wall = time.perf_counter()-offline_start\n        del embeddings\n        pd.DataFrame(gallery_rows).to_csv(out/\'gallery_timing.csv\', index=False)\n        torch.save(bank.cpu(), out/\'gallery_features.pt\')\n        # Search warmup uses the full bank, same as online queries.\n        for _ in range(config[\'warmup\']):\n            emb, _ = feature(model, transform, first, name)\n            (emb @ bank.T).argmax().item()\n        sync()\n        rows = []\n        journal = out/\'predictions.partial.csv\'\n        if journal.exists():\n            journal.unlink()  # failed attempt only; never overwrite a completed summary\n        for i, row in queries.reset_index(drop=True).iterrows():\n            image = rgb(row.uav_path)  # same I/O exclusion as previous benchmark\n            print(f\'{name}: Query {i+1}/{len(queries)}\', flush=True)\n            for repeat in range(config[\'repeats\']):\n                sync(); start = time.perf_counter()\n                emb, forward_ms = feature(model, transform, image, name)\n                sync(); search_start = time.perf_counter()\n                pick = int((emb @ bank.T).argmax().item())\n                sync(); stop = time.perf_counter()\n                g = gallery.iloc[pick]  # coordinate lookup only AFTER image-only search\n                err = distance_m(qgeo.iloc[i], g, mode) if mode else np.nan\n                record = dict(sample_id=row.sample_id, repeat=repeat, gallery_id=g.gallery_id,\n                              error_m=err, seconds=stop-start, forward_ms=forward_ms,\n                              search_ms=(stop-search_start)*1000)\n                if mode:\n                    for column in ((\'lat\',\'lon\') if mode==\'wgs84\' else (\'east_m\',\'north_m\',\'frame_id\')):\n                        record[\'pred_\'+column] = g[column]\n                        record[\'gt_\'+column] = qgeo.iloc[i][column]\n                rows.append(record)\n                pd.DataFrame([record]).to_csv(journal, mode=\'a\', index=False, header=not journal.exists())\n        df = pd.DataFrame(rows)\n        df.to_csv(out/\'predictions.csv\', index=False)\n        times = df.seconds.to_numpy()\n        first_run = df[df.repeat == 0]\n        valid_ranking = bool(config[\'position_ranking_valid\'] and mode and not smoke)\n        summary = dict(model=name, status=\'smoke_passed\' if smoke else \'measured\', n=len(queries),\n                       valid_pct=100., mean_ms=float(times.mean()*1000), p50_ms=float(np.median(times)*1000),\n                       p95_ms=float(np.percentile(times,95)*1000), fps=float(1/times.mean()),\n                       total_parameters=params, parameters_M=params/1e6,\n                       peak_allocated_MB=torch.cuda.max_memory_allocated()/1024**2,\n                       precision=\'fp32-audited\', batch_size=1, latency_repeats=config[\'repeats\'],\n                       position_ranking_valid=valid_ranking, reason=reason,\n                       latency_scope=\'preprocess+H2D+query_model+L2+GPU_cosine_top1; no image decode; precomputed GPU gallery\',\n                       gallery_n=len(gallery), gallery_precompute_wall_s=offline_wall,\n                       gallery_precompute_encode_s=sum(r[\'seconds\'] for r in gallery_rows),\n                       gallery_precompute_scope=\'wall includes decode, encode, D2H, bank upload; encode_s excludes decode/D2H\',\n                       forward_mean_ms=float(df.forward_ms.mean()), search_mean_ms=float(df.search_ms.mean()),\n                       checkpoint_sha256=digest(config[\'checkpoint\']), repo_commit=config[\'repo_commit\'],\n                       queries_sha256=digest(config[\'queries\']), gallery_sha256=digest(config[\'gallery\']),\n                       gpu=torch.cuda.get_device_name(), torch_version=torch.__version__,\n                       coordinate_provenance=config[\'coordinate_provenance\'],\n                       coordinate_distance=\'haversine WGS84\' if mode==\'wgs84\' else mode,\n                       retrieval_recall1_pct=None, retrieval_recall_reason=\'No independently verified retrieval relevance labels supplied\')\n        for threshold in (10,20,32):\n            summary[f\'accuracy_{threshold}m_pct\'] = float((first_run.error_m < threshold).mean()*100) if mode else None\n        # Mathematical distance may be computed with verified coordinates even if the original\n        # experiment is diagnostic; position_ranking_valid stays False and must accompany it.\n        summary[\'config_signature\'] = config_signature\n        result_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n        pd.DataFrame([summary]).to_csv(out/\'summary.csv\', index=False)\n        (out/\'config.json\').write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\n        return summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    args = parser.parse_args()\n    evaluate(json.loads(Path(args.config).read_text(encoding=\'utf-8\')))\n', 'localization_worker.py': "\nimport sys, json, hashlib, os\nfrom pathlib import Path\nimport torch\nfrom benchmark_core import (IMatchAdapter, BearingAdapter, benchmark, save_json)\nimport pandas as pd\nconfig=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))\nname=config['name']\nout=Path(config['output']); out.mkdir(parents=True,exist_ok=True)\nsignature=hashlib.sha256(json.dumps(config,sort_keys=True).encode()+Path(__file__).read_bytes()).hexdigest()\nresult=out/'summary.json'\nif result.exists():\n    saved=json.loads(result.read_text())\n    if saved.get('config_signature') != signature:\n        raise ValueError('Completed result uses a different protocol/weights/data. Start a new RUN.')\n    print('Preserved completed result:',result,flush=True)\n    sys.exit(0)\ntorch.cuda.set_device(0)\ntorch.backends.cuda.matmul.allow_tf32=False\ntorch.backends.cudnn.allow_tf32=False\ntorch.backends.cudnn.benchmark=False\nmanifest=pd.read_csv(config['queries'],dtype={'sample_id':str})\nif config['smoke']: manifest=manifest.head(3).copy()\nprint(name,'loading pretrained model',flush=True)\nif name=='Bearing-UAV':\n    adapter=BearingAdapter(Path(config['repo']),config['weights_dir'],'cuda:0')\nelse:\n    adapter=IMatchAdapter('cuda:0',config['angles'],config['stop_inliers'],config['checkpoint'])\nsummary=benchmark(adapter,manifest,out,config['audit'],'cuda:0',config['meters_per_unit'],\n                  1 if config['smoke'] else config['warmup'],1 if config['smoke'] else config['repeats'])\nsummary.update(config_signature=signature,smoke=config['smoke'],latency_scope='pair preprocessing + H2D + forward + geometry; no gallery cache')\nsave_json(result,summary)\nprint(name,'completed',flush=True)\n"}
for name, content in SOURCES.items():
    target=SOURCE/name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(content,encoding='utf-8')
sys.path.insert(0,str(SOURCE))
from benchmark_core import read_dataset, dataset_audit, load_pair
manifest,csv_path=read_dataset(INPUT_ROOT,SAMPLE_CSV,LIMIT,
    extract_dir=WORK/'extracted_data',fallback_root='/kaggle/input')
manifest_text=manifest.to_csv(index=False)
manifest_path=RUN/'manifest.csv'
if manifest_path.exists() and manifest_path.read_text(encoding='utf-8') != manifest_text:
    raise ValueError('This RUN has different data. Start a new RUN before changing SAMPLE_CSV/LIMIT.')
manifest_path.write_text(manifest_text,encoding='utf-8')
for _, row in manifest.iterrows(): load_pair(row)
audit=dataset_audit(manifest,METERS_PER_UNIT)
if DIAGNOSTIC_ONLY: audit['position_ranking_valid']=False
save_json(RUN/'data_audit.json',audit)
save_json(RUN/'source_hashes.json',{n:hash_file(SOURCE/n) for n in SOURCES})
print('Samples:',len(manifest),'CSV:',csv_path)
display(manifest.head()); display(pd.DataFrame([audit]))

def pair_config(name, smoke):
    prepared=json.loads((RUN/(name+'_prepared.json')).read_text(encoding='utf-8'))
    return dict(prepared,name=name,queries=str(manifest_path),queries_sha256=hash_file(manifest_path),
                source_hashes={n:hash_file(SOURCE/n) for n in SOURCES},
                audit=audit,meters_per_unit=METERS_PER_UNIT,angles=ANGLES,stop_inliers=STOP_INLIERS,
                warmup=WARMUP,repeats=REPEATS,smoke=smoke,output=str(RUN/name/('smoke' if smoke else 'full')))

[이미지 검사] 1번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k000_uav.jpg
[이미지 검사] 25번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k012_uav.jpg
[이미지 검사] 50번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k024_block.jpg
[이미지 검사] 75번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k037_uav.jpg
[이미지 검사] 100번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k049_block.jpg
[이미지 검사] 125번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k062_uav.jpg
[이미지 검사] 150번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k074_block.jpg
[이미지 검사] 175번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k087_uav.jpg
[이미지 검사] 200번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k099_block.jpg
[이미지 검사] 225번째 파일 읽는 중: /kaggle/input/datasets/heeyunju/visloc-single-eval/visloc_01/k112_uav.jpg
[이미지 검사] 250번째 파일

,sample_id,set,k,gt_x,gt_y,uav_path,uav_sha256,block_path,block_sha256
0,visloc_01:0,visloc_01,0,-0.782119,0.080590,/kaggle/input/datasets/heeyunju/visloc-single-...,1430bd166ca9364df3de590a86ef5611fb46f9ca50baaf...,/kaggle/input/datasets/heeyunju/visloc-single-...,c268a51faf8dc4b68b4c834e2e97bb2f0e8354aff45f95...
1,visloc_01:1,visloc_01,1,-0.643660,-0.536240,/kaggle/input/datasets/heeyunju/visloc-single-...,453c40525dd4b064f59a8d50658579e8fd019ea68f4a0a...,/kaggle/input/datasets/heeyunju/visloc-single-...,096ba862f27aaf97ec1fe519af450138dfaccdd6cf9ded...
2,visloc_01:2,visloc_01,2,-0.782119,0.866779,/kaggle/input/datasets/heeyunju/visloc-single-...,dcda7d554c7242e970e4876657cf2b77493bd9996ffcbe...,/kaggle/input/datasets/heeyunju/visloc-single-...,38fff102663673d54ab6aa268fabf4397b1c5f65184a59...
3,visloc_01:3,visloc_01,3,-0.747655,0.269797,/kaggle/input/datasets/heeyunju/visloc-single-...,6c6075f77c671ce40e242ccf0161170839b0c3445a0144...,/kaggle/input/datasets/heeyunju/visloc-single-...,ef586e2acdf1b9010c45df19bb6f550a670f67ba670259...
4,visloc_01:4,visloc_01,4,-0.851650,-0.307301,/kaggle/input/datasets/heeyunju/visloc-single-...,5101c24db5d2319c6c0f3d9ad7482971d4a3bc1d3c170b...,/kaggle/input/datasets/heeyunju/visloc-single-...,83391c16fdaa04772ecad3966630fdf46205500d129cbc...


,n,all_labels_at_center,center_baseline_accuracy_10m_pct,position_ranking_valid,note
0,1738,False,7.825086,False,Nonconstant labels do not prove absence of lea...


## 4. MEAN·Sample4Geo 공통 갤러리와 준비 도구

In [9]:
# 좌표가 없으면 기본값 그대로: 모든 고유 block을 고정 갤러리로 사용합니다.
RET_GALLERY_CSV = ''
RET_QUERY_GEO_CSV = ''
RET_COORDINATES_VERIFIED = False
RET_COORDINATE_PROVENANCE = ''
RET_ROOT=RUN
RET_AUDIT=audit
RET_GPU=0
ret_hash=hash_file
if RET_GALLERY_CSV:
    gp=Path(RET_GALLERY_CSV)
    gallery=pd.read_csv(gp,dtype={'gallery_id':str})
    gallery['image_path']=[str((gp.parent/str(p)).resolve()) for p in gallery.image_path]
else:
    bank={}
    for p in manifest.block_path: bank.setdefault(hash_file(p),str(p))
    gallery=pd.DataFrame([dict(gallery_id=h,image_path=p) for h,p in sorted(bank.items())])
if gallery.empty or gallery.gallery_id.duplicated().any():
    raise ValueError('Gallery must contain unique IDs')
for p in gallery.image_path:
    if not Path(p).is_file(): raise FileNotFoundError(p)
gallery['image_sha256']=[hash_file(p) for p in gallery.image_path]
gallery_text=gallery.to_csv(index=False)
if (RUN/'gallery.csv').exists() and (RUN/'gallery.csv').read_text(encoding='utf-8') != gallery_text:
    raise ValueError('Gallery changed: start a new RUN')
(RUN/'gallery.csv').write_text(gallery_text,encoding='utf-8')
manifest.to_csv(RUN/'queries.csv',index=False)
print('Queries:',len(manifest),'Gallery:',len(gallery))
print('공통 좌표 미설정 시 검색 모델의 미터 정확도/Recall@1은 N/A입니다.')
def ret_prepare(name, url, commit, checkpoint, file_id, filename, folder_url):
    repo=CACHE/('repo_'+name)
    if not (repo/'.git').exists():
        subprocess.run(['git','clone',url,str(repo)], check=True)
    subprocess.run(['git','-C',str(repo),'checkout',commit], check=True)
    if checkpoint:
        checkpoint=Path(checkpoint)
    else:
        matches=list(Path('/kaggle/input').rglob(filename))
        if len(matches)==1:
            checkpoint=matches[0]
        elif len(matches)>1:
            raise ValueError(f'Choose an exact checkpoint path: {matches}')
        else:
            destination=CACHE/(name+'_'+filename)
            if file_id:
                subprocess.run([str(RET_PY),'-m','gdown','--fuzzy',file_id,'-O',str(destination)], check=True)
            else:
                # List the official folder first, download only the exact U1652 file.
                script="import gdown, pathlib, sys\nitems=gdown.download_folder(url=sys.argv[1], skip_download=True, quiet=False)\nmatches=[x for x in (items or []) if pathlib.Path(x.path).name==sys.argv[2]]\nif len(matches)!=1:\n    raise RuntimeError('Official folder has no unique exact checkpoint. Download manually to Kaggle Dataset, then set checkpoint path. Listed entries: '+str(items))\ngdown.download(id=matches[0].id, output=sys.argv[3], quiet=False)\n"
                subprocess.run([str(RET_PY),'-c',script,folder_url,filename,str(destination)], check=True)
            checkpoint=destination
    if not checkpoint.is_file() or checkpoint.stat().st_size<1024**2:
        raise ValueError(f'Missing checkpoint or invalid/HTML download: {checkpoint}')
    config=dict(name=name,repo=str(repo),repo_commit=commit,checkpoint=str(checkpoint),checkpoint_sha256=ret_hash(checkpoint),
                queries=str(RET_ROOT/'queries.csv'),gallery=str(RET_ROOT/'gallery.csv'),
                query_geo=RET_QUERY_GEO_CSV,coordinates_verified=RET_COORDINATES_VERIFIED,
                coordinate_provenance=RET_COORDINATE_PROVENANCE,
                position_ranking_valid=bool(RET_AUDIT.get('position_ranking_valid',False)),
                gpu_index=RET_GPU,warmup=int(globals().get('WARMUP',5)),
                repeats=int(globals().get('REPEATS',3)),smoke=True,output=str(RET_ROOT/name/'smoke'))
    print(name,'checkpoint:',checkpoint,'SHA256:',ret_hash(checkpoint))
    return config


def retrieval_config(name, smoke):
    config=json.loads((RUN/(name+'_prepared.json')).read_text(encoding='utf-8'))
    config.update(smoke=smoke,output=str(RUN/name/('smoke' if smoke else 'full')),
                  warmup=WARMUP,repeats=REPEATS)
    return config

Queries: 1738 Gallery: 1738
공통 좌표 미설정 시 검색 모델의 미터 정확도/Recall@1은 N/A입니다.


## 5. Bearing-UAV — pretrained 준비 및 3장 추론

In [ ]:
BEARING_WEIGHTS_DIR = ''  # 비우면 공식 cross_view 가중치를 다운로드
BEARING_REVISION = 'main'
BEARING_PY=make_env('bearing',['huggingface_hub','imgaug==0.4.0','scikit-image','scipy'])

import zipfile
BEARING_REPO = CACHE / 'bearinguav'
BEARING_READY, BEARING_ERROR = False, ''
try:
    if not (BEARING_REPO / '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/liukejia121/bearinguav.git', str(BEARING_REPO)], check=True)
    subprocess.run(['git', '-C', str(BEARING_REPO), 'checkout', BEARING_REVISION], check=True)
    bearing_commit = subprocess.check_output(['git', '-C', str(BEARING_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if not BEARING_WEIGHTS_DIR:
        archive = subprocess.check_output([str(BEARING_PY),'-c',
            "from huggingface_hub import hf_hub_download; print(hf_hub_download('HaoyZhou/bearinguav','Bearing_UAV.zip'))"],text=True).strip().splitlines()[-1]
        destination = CACHE / 'bearing_weights'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP path')
            z.extractall(destination)
        candidates = [p.parent for p in destination.rglob('best_model.pth') if p.parent.name == 'cross_view']
        if len(candidates) != 1:
            raise ValueError(f'Select BEARING_WEIGHTS_DIR explicitly: {candidates}')
        BEARING_WEIGHTS_DIR = str(candidates[0])
    if not (Path(BEARING_WEIGHTS_DIR) / 'training_configure.json').is_file():
        raise FileNotFoundError('training_configure.json is required alongside best_model.pth')
    BEARING_READY = True
except Exception as exc:
    BEARING_ERROR = repr(exc)
    print('Bearing-UAV preparation failed:', BEARING_ERROR)

if not BEARING_READY: raise RuntimeError(BEARING_ERROR)
save_json(RUN/'Bearing-UAV_prepared.json',dict(repo=str(BEARING_REPO),repo_commit=bearing_commit,
    weights_dir=BEARING_WEIGHTS_DIR,checkpoint_sha256=hash_file(Path(BEARING_WEIGHTS_DIR)/'best_model.pth')))
bearing_smoke=run_worker('Bearing-UAV',BEARING_PY,SOURCE/'localization_worker.py',pair_config('Bearing-UAV',True))
print('Bearing-UAV pretrained 확인 완료')

In [ ]:
BEARING_WEIGHTS_DIR = ''  # 비우면 공식 cross_view 가중치를 다운로드
BEARING_REVISION = 'main'
BEARING_PY=make_env('bearing',['huggingface_hub','imgaug==0.4.0','scikit-image','scipy'])

import zipfile
BEARING_REPO = CACHE / 'bearinguav'
BEARING_READY, BEARING_ERROR = False, ''
try:
    if not (BEARING_REPO / '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/liukejia121/bearinguav.git', str(BEARING_REPO)], check=True)
    subprocess.run(['git', '-C', str(BEARING_REPO), 'checkout', BEARING_REVISION], check=True)
    bearing_commit = subprocess.check_output(['git', '-C', str(BEARING_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if not BEARING_WEIGHTS_DIR:
        archive = subprocess.check_output([str(BEARING_PY),'-c',
            "from huggingface_hub import hf_hub_download; print(hf_hub_download('HaoyZhou/bearinguav','Bearing_UAV.zip'))"],text=True).strip().splitlines()[-1]
        destination = CACHE / 'bearing_weights'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP path')
            z.extractall(destination)
        candidates = [p.parent for p in destination.rglob('best_model.pth') if p.parent.name == 'cross_view']
        if len(candidates) != 1:
            raise ValueError(f'Select BEARING_WEIGHTS_DIR explicitly: {candidates}')
        BEARING_WEIGHTS_DIR = str(candidates[0])
    if not (Path(BEARING_WEIGHTS_DIR) / 'training_configure.json').is_file():
        raise FileNotFoundError('training_configure.json is required alongside best_model.pth')
    BEARING_READY = True
except Exception as exc:
    BEARING_ERROR = repr(exc)
    print('Bearing-UAV preparation failed:', BEARING_ERROR)

if not BEARING_READY: raise RuntimeError(BEARING_ERROR)
save_json(RUN/'Bearing-UAV_prepared.json',dict(repo=str(BEARING_REPO),repo_commit=bearing_commit,
    weights_dir=BEARING_WEIGHTS_DIR,checkpoint_sha256=hash_file(Path(BEARING_WEIGHTS_DIR)/'best_model.pth')))
bearing_smoke=run_worker('Bearing-UAV',BEARING_PY,SOURCE/'localization_worker.py',pair_config('Bearing-UAV',True))
print('Bearing-UAV pretrained 확인 완료')

## 6. Bearing-UAV — 전체 평가

In [ ]:
result=run_worker('Bearing-UAV',WORK/'envs/bearing/bin/python',SOURCE/'localization_worker.py',pair_config('Bearing-UAV',False))
display(pd.DataFrame([result]))

## 7. iMatch — pretrained 준비 및 3장 추론

In [ ]:
##IMATCH_CHECKPOINT = ''  # 기본값: 우리 iMatch 파이프라인 + LoMa-B 공개 pretrained
##VISMATCH_SPEC = 'vismatch'
##IMATCH_PY=make_env('imatch',[VISMATCH_SPEC,'scipy'])
##save_json(RUN/'iMatch_prepared.json',dict(checkpoint=IMATCH_CHECKPOINT,
    ##checkpoint_sha256=hash_file(IMATCH_CHECKPOINT) if IMATCH_CHECKPOINT else 'official LoMa pretrained'))
##imatch_smoke=run_worker('iMatch',IMATCH_PY,SOURCE/'localization_worker.py',pair_config('iMatch',True))
##print('iMatch pretrained 확인 완료')

## 8. iMatch — 전체 평가

In [ ]:
##result=run_worker('iMatch',WORK/'envs/imatch/bin/python',SOURCE/'localization_worker.py',pair_config('iMatch',False))
##display(pd.DataFrame([result]))

## 9. MEAN — pretrained 준비 및 3장 추론

In [15]:
# Run once in the current Kaggle session; reuses the downloaded MEAN weights.
from pathlib import Path
worker_path = SOURCE / 'retrieval_worker.py'
worker_source = worker_path.read_text(encoding='utf-8')
helper = 'def normalize_mean_state(state, expected):\n    """Map the exact legacy training-branch namespace; never relax strict loading."""\n    suffixes = (\'FC11.weight\', \'FC12.weight\', \'FC13.weight\', \'FC1.weight\', \'FC1.bias\')\n    mapping = {\'model_1.MSE.\' + s: \'model_1.DEG.\' + s for s in suffixes}\n    legacy = {k for k in state if k.startswith(\'model_1.MSE.\')}\n    if not legacy:\n        return state\n    if legacy != set(mapping):\n        raise ValueError(\'Unrecognized MEAN MSE checkpoint keys; refusing partial conversion\')\n    converted = dict(state)\n    for old, new in mapping.items():\n        if new in state or new not in expected:\n            raise ValueError(f\'MEAN namespace collision or missing target: {old} -> {new}\')\n        if state[old].shape != expected[new].shape:\n            raise ValueError(f\'MEAN tensor shape mismatch: {old} -> {new}\')\n        converted[new] = converted.pop(old)\n    if set(converted) != set(expected):\n        raise ValueError(\'MEAN checkpoint has other missing/unexpected keys after namespace conversion\')\n    for key in converted:\n        if converted[key].shape != expected[key].shape:\n            raise ValueError(f\'MEAN tensor shape mismatch: {key}\')\n    print(\'MEAN compatibility: mapped five model_1.MSE keys to model_1.DEG; strict loading retained\', flush=True)\n    return converted\n\n\n'
if 'def normalize_mean_state(' not in worker_source:
    marker = 'def load_model(name, repo, checkpoint):'
    loader = '    model.load_state_dict(state, strict=True)'
    if worker_source.count(marker) != 1 or worker_source.count(loader) != 1:
        raise RuntimeError('Unexpected worker version; no file changed')
    worker_source = worker_source.replace(marker, helper + marker)
    worker_source = worker_source.replace(loader,
        "    if name == 'MEAN':\n        state = normalize_mean_state(state, model.state_dict())\n" + loader)
    compile(worker_source, str(worker_path), 'exec')
    backup = worker_path.with_suffix('.before_mean_keys.py')
    if not backup.exists(): backup.write_text(worker_path.read_text(encoding='utf-8'), encoding='utf-8')
    worker_path.write_text(worker_source, encoding='utf-8')
print('MEAN namespace compatibility patch ready')
result = run_worker('MEAN', WORK/'envs/mean/bin/python', worker_path, retrieval_config('MEAN', True))
print('MEAN pretrained strict load 및 추론 확인 완료')


MEAN namespace compatibility patch ready
/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:166: UserWarning: Overwriting convnext_tiny in registry with sample4geo.hand_convnext.ConvNext.backbones.model_convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:177: UserWarning: Overwriting convnext_small in registry with sample4geo.hand_convnext.ConvNext.backbones.model_convnext.convnext_small. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:188: UserWarning: Overwriting convnext_base in registry with samp

In [16]:
# Official U1652 default handcraft network: ConvNeXt-Tiny + MEAN modules, 701 classes, block=2.
# The README's default checkpoint is weights_e1_0.9239.pth. Strict mismatch fails, never substitutes a backbone.
MEAN_CHECKPOINT = '' # e.g. /kaggle/input/datasets/OWNER/WEIGHTS/weights_e1_0.9239.pth
MEAN_FILE_ID = '1QR98rrM6bFWoPDOsH29jJkFYHK6sC-f_'

RET_PY=make_env('mean',['timm==0.9.16','gdown==5.2.0','albumentations'])
prepared=ret_prepare(
    'MEAN','https://github.com/ISChenawei/MEAN.git',
    '18e183c17cdfaf73c2c32c914cc99de28ee1c2b1',MEAN_CHECKPOINT,MEAN_FILE_ID,
    'weights_e2_0.9355.pth',
    'https://drive.google.com/drive/folders/13aFkUDNzqOHAvDfaloh14RMvOuPZqi3G')
save_json(RUN/'MEAN_prepared.json',prepared)
result=run_worker('MEAN',RET_PY,SOURCE/'retrieval_worker.py',retrieval_config('MEAN',True))
print('MEAN pretrained strict load 및 추론 확인 완료')

pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)

pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


a2a-sdk==1.3.0
absl-py==2.0.0
accelerate==1.15.0
access==1.1.10.post3
affine==3.0.1
aiobotocore==2.26.0
aiofiles==25.1.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aioitertools==0.13.0
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.1
alembic==1.20.1
altair==5.5.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
anywidget==0.9.21
apsw==3.54.1.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
args==0.1.0
array_record==0.8.6
arrow==1.4.0
arviz==0.22.0
asn1crypto==1.5.1
astropy==7.2.3
astropy-iers-data==0.2026.10.19.0.52.1
asttokens==3.0.2
astunparse==1.6.3
async-lru==2.4.0
atpublic==5.1
attrs==26.1.0
audioop-lts==0.2.2
audioread==3.1.0
Authlib==1.9.1
autograd==1.9.1
babel==2.18.0
backcall==0.2.0
bayesian-optimization==3.4.0
beartype==0.23.1
beautifulsoup4==4.13.5
betterproto==2

HEAD is now at 18e183c Update README.md
Downloading...
From (original): https://drive.google.com/uc?id=1QR98rrM6bFWoPDOsH29jJkFYHK6sC-f_
From (redirected): https://drive.google.com/uc?id=1QR98rrM6bFWoPDOsH29jJkFYHK6sC-f_&confirm=t&uuid=bab23e71-343d-413e-a35c-8cf338181827
To: /kaggle/working/four_model_benchmark/pretrained/MEAN_weights_e2_0.9355.pth
100%|██████████| 146M/146M [00:03<00:00, 45.9MB/s] 


MEAN checkpoint: /kaggle/working/four_model_benchmark/pretrained/MEAN_weights_e2_0.9355.pth SHA256: bc0f0798381a7e60166ba7a2a098dfa91182aa873133fa944abcfe26e0492c4a
Preserved completed result: /kaggle/working/four_model_benchmark/run_20261114T024104439388Z/MEAN/smoke/summary.json
MEAN pretrained strict load 및 추론 확인 완료


## 10. MEAN — 전체 평가

In [20]:
result=run_worker('MEAN',WORK/'envs/mean/bin/python',SOURCE/'retrieval_worker.py',retrieval_config('MEAN',False))
display(pd.DataFrame([result]))

/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:166: UserWarning: Overwriting convnext_tiny in registry with sample4geo.hand_convnext.ConvNext.backbones.model_convnext.convnext_tiny. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:177: UserWarning: Overwriting convnext_small in registry with sample4geo.hand_convnext.ConvNext.backbones.model_convnext.convnext_small. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/kaggle/working/four_model_benchmark/pretrained/repo_MEAN/sample4geo/hand_convnext/ConvNext/backbones/model_convnext.py:188: UserWarning: Overwriting convnext_base in registry with sample4geo.hand_convnext.ConvNext.backbones.m

KeyboardInterrupt: 

## 11. Sample4Geo — pretrained 준비 및 3장 추론

In [18]:
import subprocess

check = subprocess.run(
    [
        str(WORK / "envs/sample4geo/bin/python"), "-c",
        """
import gdown
items = gdown.download_folder(
    url="https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA",
    skip_download=True,
    quiet=False,
)
print("폴더 목록:", items)
""",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(check.stdout)
print("종료 코드:", check.returncode)

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Download completed
Processing file 1dZ9itDjS-h5KHc23OQb8oHqJ3KoSaPyS pretrained.zip
폴더 목록: [GoogleDriveFileToDownload(id='1dZ9itDjS-h5KHc23OQb8oHqJ3KoSaPyS', path='pretrained.zip', local_path='/kaggle/working/ICCV2023_Sample4Geo/pretrained.zip')]

종료 코드: 0


In [19]:
# Official University-1652 ConvNeXt-Base checkpoint; do not select CVUSA/VIGOR weights.
S4G_CHECKPOINT = '' # e.g. /kaggle/input/datasets/OWNER/WEIGHTS/weights_e1_0.9515.pth
S4G_FILE_ID = ''    # optional exact Google Drive file URL or ID, from the official folder

RET_PY=make_env('sample4geo',['timm==0.9.16','gdown==5.2.0','albumentations'])
# Official bundle contains several datasets; extract only the U1652 checkpoint.
import zipfile
from pathlib import PurePosixPath

def s4g_university_member(names):
    suffix = ('university', 'convnext_base.fb_in22k_ft_in1k_384', 'weights_e1_0.9515.pth')
    matches = [name for name in names if PurePosixPath(name).parts[-3:] == suffix]
    if len(matches) != 1:
        candidates = [name for name in names if name.lower().endswith(('.pth', '.pt'))]
        raise ValueError(f'Expected one official U1652 checkpoint; found {matches}. Archive weights: {candidates}')
    return matches[0]

if not S4G_CHECKPOINT and not S4G_FILE_ID:
    S4G_ARCHIVE = CACHE/'Sample4Geo_pretrained.zip'
    if not S4G_ARCHIVE.is_file() or not zipfile.is_zipfile(S4G_ARCHIVE):
        temporary = S4G_ARCHIVE.with_suffix('.zip.partial')
        script = "import gdown,sys; result=gdown.download(id='1dZ9itDjS-h5KHc23OQb8oHqJ3KoSaPyS',output=sys.argv[1],quiet=False); assert result, 'Sample4Geo ZIP download failed'"
        subprocess.run([str(RET_PY),'-c',script,str(temporary)],check=True)
        if not zipfile.is_zipfile(temporary):
            raise ValueError(f'Download is not a ZIP: {temporary}')
        temporary.replace(S4G_ARCHIVE)
    with zipfile.ZipFile(S4G_ARCHIVE) as archive:
        member = s4g_university_member(archive.namelist())
        target = CACHE/'Sample4Geo_U1652'/'weights_e1_0.9515.pth'
        target.parent.mkdir(parents=True,exist_ok=True)
        temporary = target.with_suffix('.pth.partial')
        # A fixed destination avoids trusting archive paths; full read verifies ZIP CRC.
        with archive.open(member) as src, temporary.open('wb') as dst:
            shutil.copyfileobj(src,dst)
        temporary.replace(target)
        S4G_CHECKPOINT = str(target)
        print('Extracted official U1652 checkpoint:',member,flush=True)

prepared=ret_prepare(
    'Sample4Geo','https://github.com/Skyy93/Sample4Geo.git',
    '498c6511478053ddb4a43f24a9f4942fab238f09',S4G_CHECKPOINT,S4G_FILE_ID,
    'weights_e1_0.9515.pth',
    'https://drive.google.com/drive/folders/1PMuUqvDnCb216D8_ZDDJzDD3FxeH5BoA')
save_json(RUN/'Sample4Geo_prepared.json',prepared)
result=run_worker('Sample4Geo',RET_PY,SOURCE/'retrieval_worker.py',retrieval_config('Sample4Geo',True))
print('Sample4Geo pretrained strict load 및 추론 확인 완료')

pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)

pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


a2a-sdk==1.3.0
absl-py==2.0.0
accelerate==1.15.0
access==1.1.10.post3
affine==3.0.1
aiobotocore==2.26.0
aiofiles==25.1.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aioitertools==0.13.0
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.1
alembic==1.20.1
altair==5.5.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
anywidget==0.9.21
apsw==3.54.1.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
args==0.1.0
array_record==0.8.6
arrow==1.4.0
arviz==0.22.0
asn1crypto==1.5.1
astropy==7.2.3
astropy-iers-data==0.2026.10.19.0.52.1
asttokens==3.0.2
astunparse==1.6.3
async-lru==2.4.0
atpublic==5.1
attrs==26.1.0
audioop-lts==0.2.2
audioread==3.1.0
Authlib==1.9.1
autograd==1.9.1
babel==2.18.0
backcall==0.2.0
bayesian-optimization==3.4.0
beartype==0.23.1
beautifulsoup4==4.13.5
betterproto==2

Downloading...
From (original): https://drive.google.com/uc?id=1dZ9itDjS-h5KHc23OQb8oHqJ3KoSaPyS
From (redirected): https://drive.google.com/uc?id=1dZ9itDjS-h5KHc23OQb8oHqJ3KoSaPyS&confirm=t&uuid=92fc1181-ac75-4f8e-8524-c24c3afe9c68
To: /kaggle/working/four_model_benchmark/pretrained/Sample4Geo_pretrained.zip.partial
100%|██████████| 1.63G/1.63G [00:21<00:00, 75.1MB/s]


Extracted official U1652 checkpoint: pretrained/university/convnext_base.fb_in22k_ft_in1k_384/weights_e1_0.9515.pth


HEAD is now at 498c651 Update README.md


Sample4Geo checkpoint: /kaggle/working/four_model_benchmark/pretrained/Sample4Geo_U1652/weights_e1_0.9515.pth SHA256: 04f4c8990a8fea73171806dabfe94c405b792b3b5ef87ce2db768daa529e5332
Sample4Geo: Query 1/3
Sample4Geo: Query 2/3
Sample4Geo: Query 3/3
Sample4Geo pretrained strict load 및 추론 확인 완료


## 12. Sample4Geo — 전체 평가

In [21]:
result=run_worker('Sample4Geo',WORK/'envs/sample4geo/bin/python',SOURCE/'retrieval_worker.py',retrieval_config('Sample4Geo',False))
display(pd.DataFrame([result]))

Sample4Geo: Gallery 100/1738


KeyboardInterrupt: 

## 13. 네 모델 비교표 및 결과 ZIP 저장

In [ ]:
rows=[]
for name in MODELS:
    p=RUN/name/'full/summary.json'
    rows.append(json.loads(p.read_text(encoding='utf-8')) if p.exists() else
                dict(model=name,status='not_completed',reason='해당 모델의 준비/전체 평가 셀과 로그 확인'))
comparison=pd.DataFrame(rows)
comparison.to_csv(RUN/'comparison.csv',index=False)
columns=['model','status','n','accuracy_10m_pct','mean_ms','p50_ms','fps','parameters_M',
         'gallery_n','gallery_precompute_wall_s','position_ranking_valid','latency_scope','reason']
display(comparison.reindex(columns=columns).round(3).fillna('N/A'))
print('진단용 위치 정확도이며, pair 위치 추정과 gallery 검색의 시간 측정 범위가 다릅니다.')
# Only run artifacts are zipped. Model environments/repos/checkpoints live outside RUN.
archive=shutil.make_archive(str(RUN),'zip',RUN)
display(FileLink(archive))
print('세션 종료 전 Kaggle Save Version으로 Output을 저장하세요:',RUN)

### 공개 코드·가중치 출처
- Bearing-UAV: https://github.com/liukejia121/bearinguav · https://huggingface.co/HaoyZhou/bearinguav
- iMatch: 포함된 로컬 파이프라인, LoMa https://github.com/davnords/LoMa · https://github.com/gmberton/vismatch
- MEAN: https://github.com/ISChenawei/MEAN (University-1652 pretrained)
- Sample4Geo: https://github.com/Skyy93/Sample4Geo (University-1652 pretrained)

Google Drive 자동 다운로드가 제한되면 해당 pretrained 셀의 CHECKPOINT에
Kaggle Dataset으로 추가한 공식 pth 경로를 지정하고 그 셀만 다시 실행하세요.
로컬 검증은 셀/내장 코드 문법과 구성 검사이며 Kaggle GPU 다운로드·추론 실측은 별도입니다.